# Arrow R-CNN Common-Protocol Reimplementation

External baseline for the revised manuscript
**Topology-Aware Structural Parsing of Hand-Drawn Diagrams via Learning-Aligned Decoding**.

This notebook provides a paper-faithful common-protocol reimplementation of
Arrow R-CNN using current TorchVision. It retains the defining detector,
ResNet101-FPN backbone, arrow head/tail regression branch, optimization regime,
and graph association logic, while evaluating on the same 752/188/450
train/validation/test division and the same directed-link protocol used by the
proposed framework.

This is **not claimed as an exact reproduction of the original software**.
The historical implementation and checkpoints are not publicly available.
The public third-party implementation
https://github.com/kit-sdq/uml-diagram-recognition is used only as an
architectural cross-check.

Official dataset annotations are obtained from:
https://github.com/bernhardschaefer/handwritten-diagram-datasets


In [ ]:
# ============================================================
# MASTER CONFIG — PUBLICATION RELEASE
# ============================================================
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("IJDAR_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
WORKING_ROOT = Path(os.environ.get("IJDAR_WORKING_ROOT", PROJECT_ROOT / "workspace")).expanduser().resolve()

DATASET_ROOT = os.environ.get("IJDAR_DATASET_ROOT", str(WORKING_ROOT / "datasets_converted"))
AUTO_DISCOVER_DATASET_ROOT = True
DATASET_FAMILIES = ("fa", "fca", "fcb")
LEARNING_SOURCE_SPLITS = ("train", "val")
TEST_SPLITS = ("test",)
LEARNING_VAL_RATIO = 0.20
LEARNING_SPLIT_SEED = 39
STRICT_CANONICAL_COUNTS = True
COMMON_CANVAS_WIDTH = 1024
COMMON_CANVAS_HEIGHT = 1024
AUTO_RESIZE_IMAGES_TO_JSON_CANVAS = True
MIN_VALID_BOX_SIZE_PX = 1.0
DATASET_PREFLIGHT_SAMPLES = 48

OFFICIAL_ANNOTATION_ROOT = os.environ.get("IJDAR_OFFICIAL_ANNOTATION_ROOT", "")
AUTO_DISCOVER_OFFICIAL_ANNOTATIONS = True
ALLOW_DOWNLOAD_OFFICIAL_ANNOTATIONS = True
FALLBACK_CLONE_WHOLE_OFFICIAL_REPO = True
OFFICIAL_REPO = "https://github.com/bernhardschaefer/handwritten-diagram-datasets"
OFFICIAL_RAW_BASE = "https://raw.githubusercontent.com/bernhardschaefer/handwritten-diagram-datasets/main/datasets"
OFFICIAL_DOWNLOAD_ROOT = str(WORKING_ROOT / "arrow_rcnn_official_annotations")
OFFICIAL_FULL_REPO_ROOT = str(WORKING_ROOT / "arrow_rcnn_official_repo")

OUTPUT_ROOT = str(PROJECT_ROOT / "outputs" / "arrow_rcnn_common_protocol")

# auto -> evaluate when a checkpoint is supplied, otherwise train.
RUN_MODE = "auto"
BASELINE_CHECKPOINT = os.environ.get(
    "IJDAR_ARROW_RCNN_CHECKPOINT",
    str(PROJECT_ROOT / "trained_models" / "arrow_rcnn_common_final.pt"),
)
AUTO_DISCOVER_BASELINE_CHECKPOINT = True
RESUME_TRAINING = False
CHECKPOINT_FILENAMES = ("arrow_rcnn_common_final.pt", "arrow_rcnn_common_last.pt")

NODE_LABEL = 1
ARROW_LABEL = 2
TEXT_LABEL = 3
NUM_CLASSES_WITH_BACKGROUND = 4

USE_IMAGENET_PRETRAINED_BACKBONE = True
MODEL_MIN_SIZE = 800
MODEL_MAX_SIZE = 1333
INTERNAL_SCORE_THRESHOLD = 0.01
INTERNAL_NMS_THRESHOLD = 0.80
DETECTIONS_PER_IMAGE = 300
ARROW_PROPOSAL_IOU_THRESHOLD = 0.70
ARROW_LOSS_WEIGHT = 1.0

TRAIN_PROFILE = "paper_90k"
MAX_OPTIMIZER_STEPS_OVERRIDE = None
PHYSICAL_BATCH_SIZE = 2
EFFECTIVE_BATCH_SIZE = 4
AUTO_REDUCE_BATCH_ON_OOM = True
BASE_LR_FOR_BATCH4 = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
USE_AMP = True
NUM_DATALOADER_WORKERS = min(4, os.cpu_count() or 1)
USE_ALL_AVAILABLE_GPUS = False
MULTI_GPU_STRATEGY = "off"
DATALOADER_PIN_MEMORY = False
DATALOADER_PERSISTENT_WORKERS = False
DATALOADER_PREFETCH_FACTOR = 1

# Generic host-memory safety; independent of any notebook platform.
MEMORY_GUARD_ENABLED = True
MEMORY_CHECK_EVERY_STEPS = 50
GC_EVERY_STEPS = 250
MIN_AVAILABLE_HOST_RAM_GB = 2.0
REQUIRE_THREE_CHANNEL_IMAGES = True
CHECKPOINT_EVERY_STEPS = 1000
LOG_EVERY_STEPS = 50

USE_GEOMETRIC_AUGMENTATION = True
AUG_AFFINE_P = 0.30
AUG_ROTATE90_P = 0.30
AUG_FLIP_P = 0.30

AUTO_TUNE_SCORE_THRESHOLD = True
SCORE_THRESHOLD_CANDIDATES = (0.30, 0.40, 0.50, 0.60, 0.70, 0.80)
DEFAULT_SCORE_THRESHOLD = 0.70
REPORT_FIXED_PAPER_THRESHOLD = True
PAPER_REFERENCE_SCORE_THRESHOLD = 0.70

ARROW_ENDPOINT_CONVENTION = "slot0_tail_slot1_head"
NODE_NMS_IOU = 0.50
ARROW_NMS_IOU = 0.80
TEXT_NMS_IOU = 0.30
FA_INITIAL_ARROW_TAIL_DISTANCE_PX = 40.0

METRIC_TOLERANCE_RATIO = 0.02
NODE_IOU_ACCEPT = 0.25
COMPUTE_TDRR = True
TDRR_NODE_IOU_THRESHOLD = 0.80

RANDOM_SEED = 39
SAVE_GRAPH_PREDICTIONS = True
COMPRESS_PREDICTIONS = True
DELETE_UNCOMPRESSED_AFTER_ZIP = False
SAVE_DEBUG_OVERLAYS = True
MAX_DEBUG_OVERLAYS_PER_FAMILY = 3

QUICK_TEST = False
QUICK_TEST_TRAIN_SAMPLES = 40
QUICK_TEST_VAL_SAMPLES = 12
QUICK_TEST_TEST_SAMPLES = 18
QUICK_TEST_STEPS = 20

MANUSCRIPT_RESULTS_FILE = PROJECT_ROOT / "MANUSCRIPT_REPORTED_RESULTS.json"


## 1. Environment, dataset discovery, and official annotation acquisition

In [ ]:
import os, sys, json, math, random, shutil, subprocess, time, zipfile, csv, urllib.request, copy, gc
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
import numpy as np
import cv2
import torch
import torchvision
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou, nms

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORKING_ROOT = Path(WORKING_ROOT).expanduser().resolve()
WORKING_ROOT.mkdir(parents=True, exist_ok=True)

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("CUDA        :", torch.cuda.is_available())
print("GPU count   :", torch.cuda.device_count() if torch.cuda.is_available() else 0)
for i in range(torch.cuda.device_count() if torch.cuda.is_available() else 0):
    print(f"GPU {i:<7}: {torch.cuda.get_device_name(i)}")

def unwrap_model(model):
    return model.module if hasattr(model, "module") else model

def clear_cuda_cache_all():
    if not torch.cuda.is_available():
        return
    for _dev in range(torch.cuda.device_count()):
        with torch.cuda.device(_dev):
            torch.cuda.empty_cache()

try:
    import psutil
except Exception:
    psutil = None

try:
    cv2.setNumThreads(0)
except Exception:
    pass

def host_memory_snapshot():
    if psutil is None:
        return {
            "available_gb": float("nan"), "used_percent": float("nan"),
            "main_rss_gb": float("nan"), "child_rss_gb": float("nan"),
            "total_process_rss_gb": float("nan"),
        }
    vm = psutil.virtual_memory()
    proc = psutil.Process(os.getpid())
    main_rss = proc.memory_info().rss
    child_rss = 0
    for child in proc.children(recursive=True):
        try:
            child_rss += child.memory_info().rss
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    gb = 1024**3
    return {
        "available_gb": vm.available/gb,
        "used_percent": float(vm.percent),
        "main_rss_gb": main_rss/gb,
        "child_rss_gb": child_rss/gb,
        "total_process_rss_gb": (main_rss+child_rss)/gb,
    }

def print_host_memory(tag):
    m = host_memory_snapshot()
    print(
        f"[ram] {tag}: available={m['available_gb']:.2f} GB | "
        f"system_used={m['used_percent']:.1f}% | "
        f"main_rss={m['main_rss_gb']:.2f} GB | "
        f"children_rss={m['child_rss_gb']:.2f} GB"
    )
    return m

OUT = Path(OUTPUT_ROOT)
if QUICK_TEST:
    OUT = Path(str(OUT) + "_quicktest")
for d in (OUT, OUT/"checkpoints", OUT/"metrics", OUT/"debug"):
    d.mkdir(parents=True, exist_ok=True)
print("Output root :", OUT)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(RANDOM_SEED)

def validate_converted_root(root):
    root = Path(root)
    if not root.is_dir():
        return False, {}
    rep, total = {}, 0
    for fam in DATASET_FAMILIES:
        rep[fam] = {}
        for split in ("train", "val", "test"):
            d = root/fam/split
            idir, jdir = d/"images", d/"json"
            if not idir.is_dir() or not jdir.is_dir():
                continue
            imgs = {p.stem for p in idir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS}
            js = {p.stem for p in jdir.glob("*.json")}
            rep[fam][split] = len(imgs & js)
            total += rep[fam][split]
    return total > 0 and all((root/f).is_dir() for f in DATASET_FAMILIES), rep

def resolve_dataset_root():
    p = Path(DATASET_ROOT).expanduser()
    ok, rep = validate_converted_root(p)
    if ok:
        print("Converted dataset:", p)
        print(json.dumps(rep, indent=2))
        return p.resolve()
    roots = [PROJECT_ROOT, WORKING_ROOT, PROJECT_ROOT/"external_data"]
    if AUTO_DISCOVER_DATASET_ROOT:
        for root in roots:
            if not root.exists():
                continue
            for q in list(root.rglob("datasets_converted")) + list(root.rglob("converted_annotations")):
                ok, rep = validate_converted_root(q)
                if ok:
                    print("Auto-discovered dataset:", q)
                    print(json.dumps(rep, indent=2))
                    return q.resolve()
    raise FileNotFoundError(
        "No valid image+JSON dataset root was found. Set DATASET_ROOT or reconstruct "
        "the images from the public source repository."
    )

DATASET_ROOT_RESOLVED = resolve_dataset_root()

def resolve_baseline_checkpoint():
    raw = str(BASELINE_CHECKPOINT).strip()
    if raw:
        p = Path(raw).expanduser()
        if p.is_file():
            return p.resolve()
        if p.is_dir():
            hits = []
            for name in CHECKPOINT_FILENAMES:
                hits.extend(q.resolve() for q in p.rglob(name) if q.is_file())
            hits = sorted(set(hits), key=lambda q: (
                CHECKPOINT_FILENAMES.index(q.name) if q.name in CHECKPOINT_FILENAMES else 999,
                len(str(q)), str(q)
            ))
            if len(hits) == 1:
                return hits[0]
            if len(hits) > 1:
                raise RuntimeError("Multiple compatible checkpoints found. Set BASELINE_CHECKPOINT to the exact file.")
    if AUTO_DISCOVER_BASELINE_CHECKPOINT:
        hits = []
        for root in (PROJECT_ROOT/"trained_models", WORKING_ROOT, PROJECT_ROOT):
            if not root.exists():
                continue
            for name in CHECKPOINT_FILENAMES:
                hits.extend(q.resolve() for q in root.rglob(name) if q.is_file())
        hits = sorted(set(hits), key=lambda q: (len(str(q)), str(q)))
        if len(hits) == 1:
            return hits[0]
        if len(hits) > 1:
            raise RuntimeError("Multiple checkpoints were auto-discovered. Set BASELINE_CHECKPOINT explicitly.")
    return None

RESOLVED_BASELINE_CHECKPOINT = resolve_baseline_checkpoint()
print("Baseline checkpoint:", RESOLVED_BASELINE_CHECKPOINT)

REQUIRED_OFFICIAL_SPLITS = {
    "fa": ("train","val","test"),
    "fca": ("train","test"),
    "fcb": ("train","val","test"),
}

def looks_like_coco(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            d = json.load(f)
        return all(k in d for k in ("images","annotations","categories"))
    except Exception:
        return False

def official_root_valid(root):
    root = Path(root)
    for r in (root, root/"datasets"):
        if all(
            looks_like_coco(r/fam/f"{split}.json")
            for fam, splits in REQUIRED_OFFICIAL_SPLITS.items()
            for split in splits
        ):
            return r.resolve()
    return None

def search_official_root(root):
    root = Path(root)
    if not root.exists():
        return None
    found = []
    for p in root.rglob("train.json"):
        if p.parent.name != "fca":
            continue
        norm = official_root_valid(p.parent.parent)
        if norm is not None:
            found.append(norm)
    return sorted(set(found), key=lambda p: (len(str(p)), str(p)))[0] if found else None

def download_url(url, out, timeout=180):
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    tmp = out.with_suffix(out.suffix + ".part")
    if tmp.exists():
        tmp.unlink()
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0 ArrowRCNN-baseline"})
    with urllib.request.urlopen(req, timeout=timeout) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    tmp.replace(out)

def try_direct_annotations():
    root = Path(OFFICIAL_DOWNLOAD_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    for fam, splits in REQUIRED_OFFICIAL_SPLITS.items():
        for split in splits:
            out = root/fam/f"{split}.json"
            if looks_like_coco(out):
                continue
            url = f"{OFFICIAL_RAW_BASE}/{fam}/{split}.json"
            print("[annotations]", url)
            download_url(url, out)
    return official_root_valid(root)

def clone_full_official_repo():
    target = Path(OFFICIAL_FULL_REPO_ROOT)
    existing = search_official_root(target)
    if existing is not None:
        return existing
    if target.exists():
        shutil.rmtree(target, ignore_errors=True)
    subprocess.run(["git","clone","--depth","1",OFFICIAL_REPO,str(target)], check=True)
    norm = official_root_valid(target) or search_official_root(target)
    if norm is None:
        raise RuntimeError("Official repository does not contain the expected annotation layout.")
    return norm

def resolve_official_root():
    raw = str(OFFICIAL_ANNOTATION_ROOT).strip()
    if raw:
        norm = official_root_valid(raw)
        if norm is not None:
            return norm
    if AUTO_DISCOVER_OFFICIAL_ANNOTATIONS:
        for root in (PROJECT_ROOT, WORKING_ROOT, PROJECT_ROOT/"external_data"):
            norm = search_official_root(root)
            if norm is not None:
                return norm
    if ALLOW_DOWNLOAD_OFFICIAL_ANNOTATIONS:
        norm = try_direct_annotations()
        if norm is not None:
            return norm
    if FALLBACK_CLONE_WHOLE_OFFICIAL_REPO:
        return clone_full_official_repo()
    raise FileNotFoundError("Official annotations are required.")

OFFICIAL_ROOT_RESOLVED = resolve_official_root()
print("Official root:", OFFICIAL_ROOT_RESOLVED)
OFFICIAL_COCO = {fam:{} for fam in DATASET_FAMILIES}
for fam, splits in REQUIRED_OFFICIAL_SPLITS.items():
    for split in splits:
        with open(OFFICIAL_ROOT_RESOLVED/fam/f"{split}.json", "r", encoding="utf-8") as f:
            OFFICIAL_COCO[fam][split] = json.load(f)


## 2. Exact common sample split and topology-adapted detection records

In [ ]:
@dataclass
class Sample:
    family: str
    split: str
    sample_id: str
    image_path: Path
    json_path: Path

def find_image(jp, idir, gt):
    declared = str(gt.get("image","")).strip()
    if declared and (idir/declared).is_file(): return idir/declared
    for ext in IMAGE_EXTS:
        p = idir/(jp.stem+ext)
        if p.is_file(): return p
    return None

def discover_samples(splits):
    rows=[]
    for fam in DATASET_FAMILIES:
        for split in splits:
            sd = DATASET_ROOT_RESOLVED/fam/split; jdir,idir = sd/"json",sd/"images"
            if not jdir.is_dir() or not idir.is_dir(): continue
            for jp in sorted(jdir.glob("*.json")):
                with open(jp,"r",encoding="utf-8") as f: gt=json.load(f)
                ip=find_image(jp,idir,gt)
                if ip is None: raise FileNotFoundError(jp)
                rows.append(Sample(fam,split,jp.stem,ip,jp))
    return rows

LEARNING_POOL = discover_samples(LEARNING_SOURCE_SPLITS)
TEST_SAMPLES = discover_samples(TEST_SPLITS)

def stratified_learning_split(samples):
    by=defaultdict(list)
    for s in samples: by[s.family].append(s)
    tr=[]; va=[]; rng=random.Random(LEARNING_SPLIT_SEED)
    for fam,items in sorted(by.items()):
        items=list(items); rng.shuffle(items)
        n=max(1,int(round(len(items)*LEARNING_VAL_RATIO))) if len(items)>1 else 0
        va.extend(items[:n]); tr.extend(items[n:])
    rng.shuffle(tr); rng.shuffle(va)
    return tr,va

TRAIN_SAMPLES, VAL_SAMPLES = stratified_learning_split(LEARNING_POOL)
print("Learning pool", len(LEARNING_POOL), Counter(s.family for s in LEARNING_POOL))
print("Train", len(TRAIN_SAMPLES), Counter(s.family for s in TRAIN_SAMPLES))
print("Val", len(VAL_SAMPLES), Counter(s.family for s in VAL_SAMPLES))
print("Test", len(TEST_SAMPLES), Counter(s.family for s in TEST_SAMPLES))
if STRICT_CANONICAL_COUNTS:
    assert (len(LEARNING_POOL),len(TRAIN_SAMPLES),len(VAL_SAMPLES),len(TEST_SAMPLES)) == (940,752,188,450)
def balanced_quick_sample(samples, n, seed):
    """Take a small deterministic sample while preserving all families when n >= #families."""
    n = min(int(n), len(samples))
    if n <= 0:
        return []
    by = defaultdict(list)
    for s in samples:
        by[s.family].append(s)

    rng = random.Random(seed)
    fams = [f for f in DATASET_FAMILIES if by.get(f)]
    for f in fams:
        rng.shuffle(by[f])

    chosen = []
    if n >= len(fams):
        # Guarantee at least one sample from every family.
        for f in fams:
            chosen.append(by[f].pop())
    else:
        # If n is smaller than the number of families, choose deterministic families.
        for f in fams[:n]:
            chosen.append(by[f].pop())

    remaining = n - len(chosen)
    leftovers = []
    for f in fams:
        leftovers.extend(by[f])
    rng.shuffle(leftovers)
    chosen.extend(leftovers[:remaining])
    rng.shuffle(chosen)
    return chosen

if QUICK_TEST:
    TRAIN_SAMPLES = balanced_quick_sample(TRAIN_SAMPLES, QUICK_TEST_TRAIN_SAMPLES, RANDOM_SEED + 1)
    VAL_SAMPLES   = balanced_quick_sample(VAL_SAMPLES,   QUICK_TEST_VAL_SAMPLES,   RANDOM_SEED + 2)
    TEST_SAMPLES  = balanced_quick_sample(TEST_SAMPLES,  QUICK_TEST_TEST_SAMPLES,  RANDOM_SEED + 3)
    print("QUICK_TEST active")
    print("  train:", len(TRAIN_SAMPLES), Counter(s.family for s in TRAIN_SAMPLES))
    print("  val  :", len(VAL_SAMPLES), Counter(s.family for s in VAL_SAMPLES))
    print("  test :", len(TEST_SAMPLES), Counter(s.family for s in TEST_SAMPLES))

# Build exact filename index across official source splits.
def category_name(coco,ann):
    if "category" in ann: return str(ann["category"]).lower()
    cid=ann.get("category_id")
    for c in coco.get("categories",[]):
        if c.get("id")==cid: return str(c.get("name","")).lower()
    return ""

OFFICIAL_INDEX={fam:{} for fam in DATASET_FAMILIES}
for fam in DATASET_FAMILIES:
    for split,coco in OFFICIAL_COCO[fam].items():
        byid=defaultdict(list)
        for a in coco.get("annotations",[]): byid[int(a["image_id"])].append(a)
        for im in coco.get("images",[]):
            name=str(im["file_name"])
            if name in OFFICIAL_INDEX[fam]: raise RuntimeError(f"Duplicate official filename: {fam}/{name}")
            OFFICIAL_INDEX[fam][name]={"image":im,"annotations":byid[int(im["id"])],"coco":coco}

def load_gt_graph(sample):
    with open(sample.json_path,"r",encoding="utf-8") as f: return json.load(f)

def clip_xyxy(b,w,h):
    x1,y1,x2,y2=[float(v) for v in b]
    return [min(max(x1,0),w),min(max(y1,0),h),min(max(x2,0),w),min(max(y2,0),h)]

def detection_record(sample):
    gt=load_gt_graph(sample); W=int(gt["size"]["w"]); H=int(gt["size"]["h"])
    src=OFFICIAL_INDEX[sample.family].get(sample.image_path.name)
    if src is None: raise KeyError(f"No official annotation for {sample.family}/{sample.image_path.name}")
    im=src["image"]; sx=W/float(im["width"]); sy=H/float(im["height"])
    boxes=[]; labels=[]; kps=[]; stats=Counter()
    for a in src["annotations"]:
        name=category_name(src["coco"],a)
        label=ARROW_LABEL if name=="arrow" else TEXT_LABEL if name=="text" else NODE_LABEL
        x,y,bw,bh=[float(v) for v in a.get("bbox",[0,0,0,0])]
        box=clip_xyxy([x*sx,y*sy,(x+bw)*sx,(y+bh)*sy],W,H)
        if box[2]-box[0]<1 or box[3]-box[1]<1: continue
        kp=np.zeros((2,3),dtype=np.float32)
        if label==ARROW_LABEL:
            raw=a.get("keypoints",[]) or []
            if len(raw)<6: stats["missing_arrow_keypoints"]+=1
            else:
                kp[0]=[float(raw[0])*sx,float(raw[1])*sy,float(raw[2])]
                kp[1]=[float(raw[3])*sx,float(raw[4])*sy,float(raw[5])]
        boxes.append(box); labels.append(label); kps.append(kp)
        stats[{NODE_LABEL:"node",ARROW_LABEL:"arrow",TEXT_LABEL:"text"}[label]]+=1
    return {"sample":sample,"width":W,"height":H,"boxes":np.asarray(boxes,np.float32),"labels":np.asarray(labels,np.int64),"keypoints":np.asarray(kps,np.float32),"stats":stats}


def _point_to_xywh_box_distance(point,box):
    px,py=[float(v) for v in point]
    x,y,w,h=[float(v) for v in box]
    x2,y2=x+w,y+h
    dx=max(x-px,0.0,px-x2)
    dy=max(y-py,0.0,py-y2)
    return math.hypot(dx,dy)

def audit_official_endpoint_order():
    """
    Verify endpoint semantics directly from arrow_prev/arrow_next geometry.
    Self-loops are excluded because their source and target node are identical.
    """
    per_family={}
    decisive_total=0
    tail_head_total=0
    reversed_total=0

    for fam in DATASET_FAMILIES:
        informative=tail_head=reversed_order=ties=0
        for split,coco in OFFICIAL_COCO[fam].items():
            by_id={int(a["id"]):a for a in coco.get("annotations",[]) if "id" in a}
            for a in coco.get("annotations",[]):
                if category_name(coco,a)!="arrow":
                    continue
                if a.get("arrow_prev") is None or a.get("arrow_next") is None:
                    continue
                if int(a["arrow_prev"])==int(a["arrow_next"]):
                    continue
                raw=a.get("keypoints",[]) or []
                prev=by_id.get(int(a["arrow_prev"]))
                nxt=by_id.get(int(a["arrow_next"]))
                if len(raw)<6 or prev is None or nxt is None:
                    continue

                p0=(float(raw[0]),float(raw[1]))
                p1=(float(raw[3]),float(raw[4]))
                correct=(
                    _point_to_xywh_box_distance(p0,prev["bbox"]) +
                    _point_to_xywh_box_distance(p1,nxt["bbox"])
                )
                reversed_cost=(
                    _point_to_xywh_box_distance(p0,nxt["bbox"]) +
                    _point_to_xywh_box_distance(p1,prev["bbox"])
                )
                informative+=1
                if correct+1e-6<reversed_cost:
                    tail_head+=1
                elif reversed_cost+1e-6<correct:
                    reversed_order+=1
                else:
                    ties+=1

        per_family[fam]={
            "informative":informative,
            "slot0_tail_slot1_head":tail_head,
            "reversed":reversed_order,
            "ties":ties,
        }
        decisive_total+=tail_head+reversed_order
        tail_head_total+=tail_head
        reversed_total+=reversed_order

    support=tail_head_total/max(1,decisive_total)
    print("Official arrow endpoint-order audit:")
    for fam,v in per_family.items():
        print(" ",fam,v)
    print(
        f"  decisive support for slot0=tail/source, slot1=head/target: "
        f"{tail_head_total}/{max(1,decisive_total)} = {100*support:.3f}%"
    )

    if support<0.99:
        raise RuntimeError("Official annotation geometry does not support the configured endpoint convention.")
    if ARROW_ENDPOINT_CONVENTION!="slot0_tail_slot1_head":
        raise RuntimeError("Configured endpoint convention disagrees with the official-annotation audit.")
    return per_family

ENDPOINT_ORDER_AUDIT=audit_official_endpoint_order()

print("Prebuilding records...")
TRAIN_RECORDS=[detection_record(s) for s in TRAIN_SAMPLES]
VAL_RECORDS=[detection_record(s) for s in VAL_SAMPLES]
TEST_RECORDS=[detection_record(s) for s in TEST_SAMPLES]

def audit_image_canvas(records):
    """Check raster dimensions against each converted graph JSON canvas."""
    mismatches=[]
    unreadable=[]
    for r in records:
        p=r["sample"].image_path
        img=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
        if img is None:
            unreadable.append(str(p))
            continue
        ah,aw=img.shape[:2]
        ew,eh=int(r["width"]),int(r["height"])
        if (aw,ah)!=(ew,eh):
            mismatches.append({
                "family":r["sample"].family,
                "sample_id":r["sample"].sample_id,
                "actual":[aw,ah],
                "expected":[ew,eh],
            })
    print(f"Raster/annotation canvas audit: mismatched={len(mismatches)}, unreadable={len(unreadable)}")
    if mismatches:
        print("  first mismatches:",mismatches[:5])
        if not AUTO_RESIZE_IMAGES_TO_JSON_CANVAS:
            raise RuntimeError(
                "Raster dimensions do not match graph JSON coordinates. "
                "Enable AUTO_RESIZE_IMAGES_TO_JSON_CANVAS."
            )
        print("  Mismatched rasters will be resized to the JSON canvas before augmentation.")
    if unreadable:
        raise FileNotFoundError(f"Unreadable dataset images: {unreadable[:5]}")
    return mismatches

CANVAS_MISMATCHES=audit_image_canvas(TRAIN_RECORDS+VAL_RECORDS+TEST_RECORDS)

audit=Counter()
for r in TRAIN_RECORDS+VAL_RECORDS+TEST_RECORDS: audit.update(r["stats"])
print("Annotation audit:", audit)
if audit["missing_arrow_keypoints"]: raise RuntimeError("Arrow keypoints missing from official annotations")

## 3. Dataset augmentation and Arrow R-CNN model

In [ ]:

def transform_points_affine(points,M):
    if len(points)==0:
        return points
    xy1=np.concatenate(
        [points.astype(np.float32),np.ones((len(points),1),dtype=np.float32)],
        axis=1
    )
    return xy1@M.T

def sanitize_instances(boxes,labels,keypoints,w,h,min_size=MIN_VALID_BOX_SIZE_PX):
    """
    Canonicalize one detection target after any geometric transform.
    Boxes are clipped to the current raster; non-finite / degenerate boxes are removed.
    Labels and keypoints are filtered with exactly the same mask.
    """
    boxes=np.asarray(boxes,dtype=np.float32).reshape(-1,4)
    labels=np.asarray(labels,dtype=np.int64).reshape(-1)
    keypoints=np.asarray(keypoints,dtype=np.float32).reshape(-1,2,3)

    if not (len(boxes)==len(labels)==len(keypoints)):
        raise RuntimeError(
            f"Target length mismatch: boxes={len(boxes)}, labels={len(labels)}, keypoints={len(keypoints)}"
        )

    if len(boxes)==0:
        return boxes,labels,keypoints,0

    boxes[:,0]=np.clip(boxes[:,0],0,float(w))
    boxes[:,2]=np.clip(boxes[:,2],0,float(w))
    boxes[:,1]=np.clip(boxes[:,1],0,float(h))
    boxes[:,3]=np.clip(boxes[:,3],0,float(h))

    finite_boxes=np.isfinite(boxes).all(axis=1)
    widths=boxes[:,2]-boxes[:,0]
    heights=boxes[:,3]-boxes[:,1]
    keep=finite_boxes & (widths>=float(min_size)) & (heights>=float(min_size))

    # Any non-finite endpoint is made invisible instead of poisoning the arrow loss.
    finite_kp=np.isfinite(keypoints[:,:,:2]).all(axis=2)
    bad_kp=~finite_kp
    keypoints[bad_kp,2]=0.0
    keypoints[bad_kp,0]=0.0
    keypoints[bad_kp,1]=0.0

    removed=int((~keep).sum())
    return boxes[keep],labels[keep],keypoints[keep],removed

def transform_boxes_affine(boxes,M,w,h):
    if len(boxes)==0:
        return np.zeros((0,4),dtype=np.float32)
    out=[]
    for b in boxes:
        x1,y1,x2,y2=[float(v) for v in b]
        corners=np.array([[x1,y1],[x2,y1],[x2,y2],[x1,y2]],dtype=np.float32)
        t=transform_points_affine(corners,M)
        out.append([
            float(t[:,0].min()),float(t[:,1].min()),
            float(t[:,0].max()),float(t[:,1].max())
        ])
    return np.asarray(out,dtype=np.float32)

def apply_affine(img,boxes,keypoints,angle,scale,tx,ty):
    h,w=img.shape[:2]
    M=cv2.getRotationMatrix2D((w/2.0,h/2.0),float(angle),float(scale)).astype(np.float32)
    M[0,2]+=float(tx)
    M[1,2]+=float(ty)
    img2=cv2.warpAffine(
        img,M,(w,h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255,255,255)
    )
    boxes2=transform_boxes_affine(boxes,M,w,h)
    kp=keypoints.copy()
    for i in range(len(kp)):
        vis=kp[i,:,2]>0
        if vis.any():
            kp[i,vis,:2]=transform_points_affine(kp[i,vis,:2],M)
            inside=(
                (kp[i,:,0]>=0)&(kp[i,:,0]<=w)&
                (kp[i,:,1]>=0)&(kp[i,:,1]<=h)
            )
            kp[i,~inside,2]=0
    return img2,boxes2,kp

def apply_rot90(img,boxes,keypoints,k):
    """Rotate raster, boxes, and keypoints by k*90° CCW; supports rectangular images."""
    k=int(k)%4
    if k==0:
        return img,boxes,keypoints
    h,w=img.shape[:2]
    img2=np.ascontiguousarray(np.rot90(img,k))
    def tp(x,y):
        if k==1:
            return y,w-x
        if k==2:
            return w-x,h-y
        return h-y,x

    out=[]
    for b in boxes:
        x1,y1,x2,y2=[float(v) for v in b]
        c=np.asarray([tp(x1,y1),tp(x2,y1),tp(x2,y2),tp(x1,y2)],dtype=np.float32)
        out.append([c[:,0].min(),c[:,1].min(),c[:,0].max(),c[:,1].max()])
    boxes2=np.asarray(out,dtype=np.float32).reshape(-1,4)

    kp=keypoints.copy()
    for i in range(len(kp)):
        for j in range(2):
            if kp[i,j,2]>0:
                kp[i,j,0],kp[i,j,1]=tp(float(kp[i,j,0]),float(kp[i,j,1]))
    return img2,boxes2,kp

def apply_random_flip(img,boxes,keypoints):
    h,w=img.shape[:2]
    mode=random.choice(("h","v","hv"))
    b=boxes.copy()
    kp=keypoints.copy()
    if "h" in mode:
        img=np.ascontiguousarray(img[:,::-1])
        x1=b[:,0].copy(); x2=b[:,2].copy()
        b[:,0]=w-x2; b[:,2]=w-x1
        vis=kp[:,:,2]>0
        kp[:,:,0]=np.where(vis,w-kp[:,:,0],kp[:,:,0])
    if "v" in mode:
        img=np.ascontiguousarray(img[::-1,:])
        y1=b[:,1].copy(); y2=b[:,3].copy()
        b[:,1]=h-y2; b[:,3]=h-y1
        vis=kp[:,:,2]>0
        kp[:,:,1]=np.where(vis,h-kp[:,:,1],kp[:,:,1])
    return img,b,kp

def resize_raster_to_record_canvas(img,record):
    target_w,target_h=int(record["width"]),int(record["height"])
    h,w=img.shape[:2]
    if (w,h)==(target_w,target_h):
        return img
    if not AUTO_RESIZE_IMAGES_TO_JSON_CANVAS:
        raise RuntimeError(
            f"Raster/JSON canvas mismatch for {record['sample'].image_path.name}: "
            f"raster={w}x{h}, json={target_w}x{target_h}"
        )
    interp=cv2.INTER_AREA if (w>target_w or h>target_h) else cv2.INTER_CUBIC
    return cv2.resize(img,(target_w,target_h),interpolation=interp)

class ArrowRCNNDataset(Dataset):
    def __init__(self,records,augment=False):
        self.records=list(records)
        self.augment=bool(augment)

    def __len__(self):
        return len(self.records)

    def __getitem__(self,idx):
        r=self.records[idx]
        img=cv2.imread(str(r["sample"].image_path),cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(r["sample"].image_path)
        img=resize_raster_to_record_canvas(img,r)
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

        boxes=r["boxes"].copy()
        labels=r["labels"].copy()
        kp=r["keypoints"].copy()

        # Always sanitize source annotations against the actual training raster.
        h,w=img.shape[:2]
        boxes,labels,kp,_=sanitize_instances(boxes,labels,kp,w,h)

        if self.augment and USE_GEOMETRIC_AUGMENTATION and len(boxes):
            original=(img.copy(),boxes.copy(),labels.copy(),kp.copy())

            if random.random()<AUG_AFFINE_P:
                img,boxes,kp=apply_affine(
                    img,boxes,kp,
                    random.uniform(-5.0,5.0),
                    random.uniform(0.80,1.00),
                    random.uniform(-0.01,0.01)*w,
                    random.uniform(-0.01,0.01)*h
                )
                h,w=img.shape[:2]
                boxes,labels,kp,_=sanitize_instances(boxes,labels,kp,w,h)

            if len(boxes) and random.random()<AUG_ROTATE90_P:
                img,boxes,kp=apply_rot90(img,boxes,kp,random.choice((1,2,3)))
                h,w=img.shape[:2]
                boxes,labels,kp,_=sanitize_instances(boxes,labels,kp,w,h)

            if len(boxes) and random.random()<AUG_FLIP_P:
                img,boxes,kp=apply_random_flip(img,boxes,kp)
                h,w=img.shape[:2]
                boxes,labels,kp,_=sanitize_instances(boxes,labels,kp,w,h)

            # A pathological augmentation should not turn a valid diagram into an empty target.
            if len(boxes)==0 and len(original[1])>0:
                img,boxes,labels,kp=original

        h,w=img.shape[:2]
        boxes,labels,kp,_=sanitize_instances(boxes,labels,kp,w,h)

        # Hard invariant: torchvision Faster R-CNN must never receive a degenerate box.
        if len(boxes):
            wh=boxes[:,2:4]-boxes[:,0:2]
            if not np.isfinite(boxes).all() or not np.all(wh>0):
                raise RuntimeError(
                    f"Invalid target survived sanitization for {r['sample'].family}/{r['sample'].sample_id}: "
                    f"{boxes[(~np.isfinite(boxes).all(axis=1)) | (wh[:,0]<=0) | (wh[:,1]<=0)][:3]}"
                )

        image=torch.from_numpy(np.ascontiguousarray(img)).permute(2,0,1).float()/255.0
        target={
            "boxes":torch.as_tensor(boxes,dtype=torch.float32),
            "labels":torch.as_tensor(labels,dtype=torch.int64),
            "keypoints":torch.as_tensor(kp,dtype=torch.float32),
            "image_id":torch.tensor([idx],dtype=torch.int64),
        }
        return image,target,r["sample"]

def collate(batch):
    return tuple(zip(*batch))

def worker_seed(worker_id):
    seed=RANDOM_SEED+worker_id
    random.seed(seed)
    np.random.seed(seed)
    try:
        cv2.setNumThreads(0)
    except Exception:
        pass
    try:
        torch.set_num_threads(1)
    except Exception:
        pass

def make_loader(dataset,batch_size,shuffle):
    workers=min(NUM_DATALOADER_WORKERS,os.cpu_count() or 1)
    kwargs=dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=bool(DATALOADER_PIN_MEMORY and torch.cuda.is_available()),
        collate_fn=collate,
        worker_init_fn=worker_seed,
    )
    if workers>0:
        kwargs["persistent_workers"]=bool(DATALOADER_PERSISTENT_WORKERS)
        kwargs["prefetch_factor"]=int(DATALOADER_PREFETCH_FACTOR)
    return DataLoader(**kwargs)

TRAIN_DATASET=ArrowRCNNDataset(TRAIN_RECORDS,True)
VAL_DATASET=ArrowRCNNDataset(VAL_RECORDS,False)
TEST_DATASET=ArrowRCNNDataset(TEST_RECORDS,False)

def preflight_dataset_targets(dataset,n_samples=DATASET_PREFLIGHT_SAMPLES,repeats=3):
    """Exercise real dataset loading + augmentation before any long training starts."""
    if len(dataset)==0:
        raise RuntimeError("Training dataset is empty.")
    rng=random.Random(RANDOM_SEED+991)
    indices=list(range(len(dataset)))
    if len(indices)>n_samples:
        indices=rng.sample(indices,n_samples)
    checked=0
    for _ in range(max(1,int(repeats))):
        for idx in indices:
            image,target,_=dataset[idx]
            b=target["boxes"]
            if b.numel():
                wh=b[:,2:4]-b[:,0:2]
                assert torch.isfinite(b).all()
                assert (wh>0).all(), b[(wh[:,0]<=0)|(wh[:,1]<=0)]
                assert (b[:,0]>=0).all() and (b[:,1]>=0).all()
                assert (b[:,2]<=image.shape[2]+1e-4).all()
                assert (b[:,3]<=image.shape[1]+1e-4).all()
            assert len(target["boxes"])==len(target["labels"])==len(target["keypoints"])
            checked+=1
    print(f"Dataset augmentation preflight: PASS ({checked} augmented samples checked)")

from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models import ResNet101_Weights
from torchvision.models.detection.roi_heads import RoIHeads, fastrcnn_loss

class ArrowRoIHeads(RoIHeads):
    def __init__(self,*args,arrow_label=ARROW_LABEL,endpoint_in_features=1024,arrow_loss_weight=1.0,arrow_train_iou=.70,**kwargs):
        super().__init__(*args,**kwargs)
        self.arrow_label=int(arrow_label); self.arrow_loss_weight=float(arrow_loss_weight); self.arrow_train_iou=float(arrow_train_iou)
        self.arrow_keypoint_pred=nn.Linear(int(endpoint_in_features),4)
        nn.init.normal_(self.arrow_keypoint_pred.weight,std=.001); nn.init.constant_(self.arrow_keypoint_pred.bias,0)
    @staticmethod
    def normalize_arrow_keypoints(kp,boxes):
        cx=(boxes[:,0]+boxes[:,2])*.5; cy=(boxes[:,1]+boxes[:,3])*.5
        hw=(boxes[:,2]-boxes[:,0]).clamp(min=1e-6)*.5; hh=(boxes[:,3]-boxes[:,1]).clamp(min=1e-6)*.5
        out=torch.empty((len(boxes),4),device=boxes.device,dtype=boxes.dtype)
        out[:,0]=(kp[:,0,0]-cx)/hw; out[:,1]=(kp[:,0,1]-cy)/hh; out[:,2]=(kp[:,1,0]-cx)/hw; out[:,3]=(kp[:,1,1]-cy)/hh
        return out
    @staticmethod
    def decode_arrow_keypoints(pred,boxes):
        cx=(boxes[:,0]+boxes[:,2])*.5; cy=(boxes[:,1]+boxes[:,3])*.5
        hw=(boxes[:,2]-boxes[:,0]).clamp(min=1e-6)*.5; hh=(boxes[:,3]-boxes[:,1]).clamp(min=1e-6)*.5
        kp=torch.zeros((len(boxes),2,3),device=boxes.device,dtype=boxes.dtype)
        kp[:,0,0]=cx+pred[:,0]*hw; kp[:,0,1]=cy+pred[:,1]*hh; kp[:,1,0]=cx+pred[:,2]*hw; kp[:,1,1]=cy+pred[:,3]*hh; kp[:,:,2]=1
        return kp
    def arrow_regression_loss(self,arrow_pred,proposals,matched_idxs,labels,targets):
        vals=[]; off=0
        for props,midx,labs,targ in zip(proposals,matched_idxs,labels,targets):
            n_=len(props); pred_i=arrow_pred[off:off+n_]; off+=n_; pos=torch.where(labs==self.arrow_label)[0]
            if len(pos)==0: continue
            matched=midx[pos]; gt_boxes=targ["boxes"][matched]; gt_kp=targ["keypoints"][matched]
            pair_iou=torch.diag(box_iou(props[pos],gt_boxes)); xy=gt_kp[:,:,:2]; visible=(gt_kp[:,:,2]>0).all(1); pbox=props[pos]
            inside=((xy[:,:,0]>=pbox[:,None,0])&(xy[:,:,0]<=pbox[:,None,2])&(xy[:,:,1]>=pbox[:,None,1])&(xy[:,:,1]<=pbox[:,None,3])).all(1)
            keep=(pair_iou>=self.arrow_train_iou)&visible&inside
            if keep.any(): vals.append(F.mse_loss(pred_i[pos[keep]],self.normalize_arrow_keypoints(xy[keep],pbox[keep]),reduction="mean"))
        return torch.stack(vals).mean()*self.arrow_loss_weight if vals else arrow_pred.sum()*0
    def forward(self,features,proposals,image_shapes,targets=None):
        if self.training: proposals,matched_idxs,labels,reg_targets=self.select_training_samples(proposals,targets)
        else: matched_idxs=labels=reg_targets=None
        bf=self.box_roi_pool(features,proposals,image_shapes); bf=self.box_head(bf); logits,boxreg=self.box_predictor(bf)
        result=[]; losses={}
        if self.training:
            lc,lb=fastrcnn_loss(logits,boxreg,labels,reg_targets); ap=self.arrow_keypoint_pred(bf); la=self.arrow_regression_loss(ap,proposals,matched_idxs,labels,targets)
            losses={"loss_classifier":lc,"loss_box_reg":lb,"loss_arrow_reg":la}
        else:
            boxes,scores,labs=self.postprocess_detections(logits,boxreg,proposals,image_shapes)
            for b,s,l in zip(boxes,scores,labs): result.append({"boxes":b,"scores":s,"labels":l})
            arrow_boxes=[]; arrow_indices=[]
            for r in result:
                idx=torch.where(r["labels"]==self.arrow_label)[0]; arrow_indices.append(idx); arrow_boxes.append(r["boxes"][idx])
            total=sum(len(x) for x in arrow_boxes)
            if total:
                af=self.box_roi_pool(features,arrow_boxes,image_shapes); af=self.box_head(af); pred=self.arrow_keypoint_pred(af); off=0
                for r,idx,bx in zip(result,arrow_indices,arrow_boxes):
                    allkp=torch.zeros((len(r["boxes"]),2,3),device=r["boxes"].device,dtype=r["boxes"].dtype); n_=len(bx)
                    if n_: allkp[idx]=self.decode_arrow_keypoints(pred[off:off+n_],bx)
                    r["keypoints"]=allkp; off+=n_
            else:
                for r in result: r["keypoints"]=torch.zeros((len(r["boxes"]),2,3),device=r["boxes"].device,dtype=r["boxes"].dtype)
        return result,losses

def build_arrow_rcnn(pretrained=True):
    weights=ResNet101_Weights.DEFAULT if pretrained else None
    backbone=resnet_fpn_backbone(backbone_name="resnet101",weights=weights,trainable_layers=5)
    model=FasterRCNN(backbone,num_classes=NUM_CLASSES_WITH_BACKGROUND,min_size=MODEL_MIN_SIZE,max_size=MODEL_MAX_SIZE,rpn_pre_nms_top_n_train=2000,rpn_post_nms_top_n_train=2000,rpn_pre_nms_top_n_test=1000,rpn_post_nms_top_n_test=1000,rpn_nms_thresh=.70,rpn_fg_iou_thresh=.70,rpn_bg_iou_thresh=.30,box_score_thresh=INTERNAL_SCORE_THRESHOLD,box_nms_thresh=INTERNAL_NMS_THRESHOLD,box_detections_per_img=DETECTIONS_PER_IMAGE)
    old=model.roi_heads; feat=old.box_predictor.cls_score.in_features
    model.roi_heads=ArrowRoIHeads(old.box_roi_pool,old.box_head,old.box_predictor,old.proposal_matcher.high_threshold,old.proposal_matcher.low_threshold,old.fg_bg_sampler.batch_size_per_image,old.fg_bg_sampler.positive_fraction,old.box_coder.weights,old.score_thresh,old.nms_thresh,old.detections_per_img,arrow_label=ARROW_LABEL,endpoint_in_features=feat,arrow_loss_weight=ARROW_LOSS_WEIGHT,arrow_train_iou=ARROW_PROPOSAL_IOU_THRESHOLD)
    return model

# Critical regression test for paper endpoint encoding: half-width/half-height.
b=torch.tensor([[0.,0.,100.,200.]]); kp=torch.tensor([[[100.,200.],[0.,0.]]]); enc=ArrowRoIHeads.normalize_arrow_keypoints(kp,b)
assert torch.allclose(enc,torch.tensor([[1.,1.,-1.,-1.]]),atol=1e-6)
assert torch.allclose(ArrowRoIHeads.decode_arrow_keypoints(enc,b)[:,:,:2],kp,atol=1e-6)
print("Endpoint encoding regression test: PASS")

In [ ]:
# ============================================================
# Geometry + target-sanitization regression tests
# ============================================================
_test_img=np.zeros((100,200,3),dtype=np.uint8)
_test_boxes=np.array([
    [10.,20.,60.,70.],
    [199.,30.,200.,60.],  # thin boundary box
],dtype=np.float32)
_test_labels=np.array([NODE_LABEL,ARROW_LABEL],dtype=np.int64)
_test_kp=np.array([
    [[0.,0.,0.],[0.,0.,0.]],
    [[199.,40.,2.],[199.5,50.,2.]],
],dtype=np.float32)

for _k in (1,2,3):
    _im,_b,_kp=apply_rot90(_test_img,_test_boxes,_test_kp,_k)
    _hh,_ww=_im.shape[:2]
    _b,_l,_kp,_removed=sanitize_instances(_b,_test_labels,_kp,_ww,_hh)
    if len(_b):
        _wh=_b[:,2:4]-_b[:,0:2]
        assert np.isfinite(_b).all()
        assert np.all(_wh>0)
        assert np.all(_b[:,0]>=0) and np.all(_b[:,1]>=0)
        assert np.all(_b[:,2]<=_ww+1e-5) and np.all(_b[:,3]<=_hh+1e-5)

# Explicit degenerate-box test: must be removed, never forwarded to Faster R-CNN.
_b=np.array([[10.,10.,10.,20.],[5.,5.,20.,20.]],dtype=np.float32)
_l=np.array([NODE_LABEL,NODE_LABEL],dtype=np.int64)
_kp=np.zeros((2,2,3),dtype=np.float32)
_b,_l,_kp,_removed=sanitize_instances(_b,_l,_kp,100,100)
assert _removed==1 and len(_b)==1
assert (_b[:,2]>_b[:,0]).all() and (_b[:,3]>_b[:,1]).all()

preflight_dataset_targets(TRAIN_DATASET)

print("Geometry/target regression tests: PASS")


## 4. Postprocessing, common evaluator, model build, and pre-training pipeline check

Before any long training begins, the notebook defines the final graph-conversion
and evaluation path, builds the model, and runs an end-to-end inference →
graph JSON → common-metric preflight. This is intentional: late-stage evaluator
or serialization errors should be caught before a 30k-step GPU run starts.

`strong_30k` is the default 30k-step profile. `paper_90k` uses the published
90k-step schedule. Resume is **opt-in** via `RESUME_TRAINING=True`; smoke-test
checkpoints are stored in a different output directory and cannot silently
contaminate the full run.

In [ ]:
def point_box_distance(point,box):
    x,y=point; x1,y1,x2,y2=[float(v) for v in box]
    dx=max(x1-x,0.0,x-x2); dy=max(y1-y,0.0,y-y2)
    return math.hypot(dx,dy)

def class_nms(boxes,scores,threshold):
    if len(boxes)==0: return torch.empty((0,),dtype=torch.long)
    return nms(boxes,scores,float(threshold))

def filter_raw_detection(raw,score_threshold):
    keep=raw["scores"]>=float(score_threshold)
    return {k:v[keep] for k,v in raw.items() if k in {"boxes","scores","labels","keypoints"}}

def graph_from_detection(raw,sample,score_threshold):
    det=filter_raw_detection(raw,score_threshold)
    boxes,scores,labels,keypoints=det["boxes"],det["scores"],det["labels"],det["keypoints"]

    keep_global=[]
    for lab,thr in [(NODE_LABEL,NODE_NMS_IOU),(ARROW_LABEL,ARROW_NMS_IOU),(TEXT_LABEL,TEXT_NMS_IOU)]:
        idx=torch.where(labels==lab)[0]
        if len(idx): keep_global.extend(idx[class_nms(boxes[idx],scores[idx],thr)].tolist())
    keep_global=sorted(keep_global,key=lambda i:float(scores[i]),reverse=True)
    if keep_global:
        ii=torch.tensor(keep_global,dtype=torch.long)
        boxes,scores,labels,keypoints=boxes[ii],scores[ii],labels[ii],keypoints[ii]
    else:
        boxes,scores,labels,keypoints=boxes[:0],scores[:0],labels[:0],keypoints[:0]

    node_idx=torch.where(labels==NODE_LABEL)[0]; arrow_idx=torch.where(labels==ARROW_LABEL)[0]
    nodes=[]; node_boxes=[]
    for ni,idx in enumerate(node_idx.tolist()):
        x1,y1,x2,y2=[float(v) for v in boxes[idx]]; node_boxes.append([x1,y1,x2,y2])
        nodes.append({"id":f"N{ni}","bbox":[x1,y1,x2-x1,y2-y1],
                      "center":{"x":(x1+x2)/2,"y":(y1+y2)/2},"score":float(scores[idx])})

    candidates=[]
    for idx in arrow_idx.tolist():
        kp=keypoints[idx]
        # Official geometry audit: slot 0 is source/tail; slot 1 is target/head.
        # The existing 30k checkpoint was trained on this raw slot order.
        tail=(float(kp[0,0]),float(kp[0,1]))
        head=(float(kp[1,0]),float(kp[1,1]))
        head_node=tail_node=None; head_dist=tail_dist=float("inf")
        if node_boxes:
            hd=[point_box_distance(head,b) for b in node_boxes]; td=[point_box_distance(tail,b) for b in node_boxes]
            hi=int(np.argmin(hd)); ti=int(np.argmin(td)); head_node=nodes[hi]["id"]; head_dist=float(hd[hi]); tail_dist=float(td[ti])
            if sample.family=="fa" and tail_dist>FA_INITIAL_ARROW_TAIL_DISTANCE_PX: tail_node=None
            else: tail_node=nodes[ti]["id"]
        candidates.append({"tail_node_id":tail_node,"head_node_id":head_node,
                           "tail_hint":{"x":tail[0],"y":tail[1]},"head_hint":{"x":head[0],"y":head[1]},
                           "confidence":float(scores[idx]),"_tail_distance":tail_dist,"_head_distance":head_dist})

    best={}
    for e in candidates:
        pair=(e["tail_node_id"],e["head_node_id"])
        if pair not in best or e["confidence"]>best[pair]["confidence"]: best[pair]=e
    arrows=[]
    for ei,e in enumerate(sorted(best.values(),key=lambda x:x["confidence"],reverse=True),1):
        arrows.append({"id":ei,"tail_node_id":e["tail_node_id"],"head_node_id":e["head_node_id"],
                       "tail_hint":e["tail_hint"],"head_hint":e["head_hint"],"confidence":e["confidence"]})

    with open(sample.json_path,"r",encoding="utf-8") as f: gt=json.load(f)
    return {"image":sample.image_path.name,"size":{"w":int(gt["size"]["w"]),"h":int(gt["size"]["h"])},
            "nodes":nodes,"arrows":arrows,"coordinate_space":"common_1024_graph_space",
            "metadata":{"method":"Arrow R-CNN reimplementation","score_threshold":float(score_threshold),
                        "node_nms_iou":NODE_NMS_IOU,"arrow_nms_iou":ARROW_NMS_IOU,"text_nms_iou":TEXT_NMS_IOU,
                        "fa_tail_threshold_px":FA_INITIAL_ARROW_TAIL_DISTANCE_PX,
                        "endpoint_semantics":"slot0=tail/source, slot1=head/target"}}

def reverse_predicted_graph_direction(pred):
    """Diagnostic only: reverse all directed edges without rerunning the detector."""
    q=copy.deepcopy(pred)
    for e in q.get("arrows",[]) or []:
        e["tail_node_id"],e["head_node_id"]=e.get("head_node_id"),e.get("tail_node_id")
        e["tail_hint"],e["head_hint"]=e.get("head_hint"),e.get("tail_hint")
    q.setdefault("metadata",{})["diagnostic_direction_reversed"]=True
    return q


In [ ]:
def safe_div(a,b): return float(a)/float(b) if float(b)!=0 else 0.0

def prf(tp,fp,fn):
    p=safe_div(tp,tp+fp); r=safe_div(tp,tp+fn); return p,r,safe_div(2*p*r,p+r)

def l2(a,b): return math.hypot(float(a[0])-float(b[0]),float(a[1])-float(b[1]))

def bbox_iou_xywh(a,b):
    ax,ay,aw,ah=[float(v) for v in a]; bx,by,bw,bh=[float(v) for v in b]
    ix1,iy1=max(ax,bx),max(ay,by); ix2,iy2=min(ax+aw,bx+bw),min(ay+ah,by+bh)
    iw,ih=max(0.,ix2-ix1),max(0.,iy2-iy1); inter=iw*ih; union=max(0.,aw)*max(0.,ah)+max(0.,bw)*max(0.,bh)-inter
    return safe_div(inter,union)

def bbox_center(node):
    x,y,w,h=[float(v) for v in node.get("bbox",[0,0,0,0])]; return x+w/2,y+h/2

def metric_tolerance_px(width,height): return int(max(2,round(METRIC_TOLERANCE_RATIO*max(int(width),int(height)))))

def tolerant_match_nodes(pred_nodes,gt_nodes,width,height):
    tol=metric_tolerance_px(width,height); candidates=[]
    for pi,pn in enumerate(pred_nodes):
        pc=bbox_center(pn)
        for gi,gn in enumerate(gt_nodes):
            gc=bbox_center(gn); iou=bbox_iou_xywh(pn["bbox"],gn["bbox"]); cd=l2(pc,gc); gb=gn["bbox"]
            center_tol=max(float(tol),.18*math.hypot(float(gb[2]),float(gb[3])))
            if iou>=NODE_IOU_ACCEPT or cd<=center_tol:
                candidates.append((cd/max(center_tol,1e-6)+(1-iou),-iou,cd,pi,gi,iou))
    candidates.sort(); up=set(); ug=set(); mapping={}; matches=[]
    for _,_,cd,pi,gi,iou in candidates:
        if pi in up or gi in ug: continue
        up.add(pi); ug.add(gi); pid=str(pred_nodes[pi].get("id",f"P{pi}")); gid=str(gt_nodes[gi].get("id",f"G{gi}"))
        mapping[pid]=gid; matches.append((pid,gid,float(iou),float(cd)))
    return mapping,matches

def normalize_node_id(v):
    return None if v is None else str(v)

def edge_pair(e):
    return normalize_node_id(e.get("tail_node_id")),normalize_node_id(e.get("head_node_id"))

def mapped_edge_pair(e,mapping):
    t=e.get("tail_node_id"); h=e.get("head_node_id")
    return (mapping.get(str(t)) if t is not None else None,mapping.get(str(h)) if h is not None else None)

def point_from_edge(e,which):
    p=e.get(which)
    if p is None: p=e.get(which+"_hint")
    if isinstance(p,dict) and "x" in p and "y" in p: return float(p["x"]),float(p["y"])
    if isinstance(p,(list,tuple)) and len(p)>=2: return float(p[0]),float(p[1])
    return None

def counter_intersection(a,b): return int(sum(min(v,b.get(k,0)) for k,v in a.items()))

def pair_matched_edges(pred_edges,gt_edges,pred_pairs,gt_pairs):
    gt_by=defaultdict(list)
    for gi,p in enumerate(gt_pairs): gt_by[p].append(gi)
    used=set(); out=[]
    for pe,pair in zip(pred_edges,pred_pairs):
        choices=[i for i in gt_by.get(pair,[]) if i not in used]
        if not choices: continue
        pt,ph=point_from_edge(pe,"tail"),point_from_edge(pe,"head"); best=None; bestd=float("inf")
        for gi in choices:
            ge=gt_edges[gi]; gt,gh=point_from_edge(ge,"tail"),point_from_edge(ge,"head"); ds=[]
            if pt is not None and gt is not None: ds.append(l2(pt,gt))
            if ph is not None and gh is not None: ds.append(l2(ph,gh))
            d=float(np.mean(ds)) if ds else 0.
            if d<bestd: bestd=d; best=gi
        used.add(best); out.append((pe,gt_edges[best]))
    return out

def evaluate_common(pred,gt):
    w,h=int(gt["size"]["w"]),int(gt["size"]["h"]); pn=pred.get("nodes",[]) or []; gn=gt.get("nodes",[]) or []
    pe=pred.get("arrows",[]) or []; ge=gt.get("arrows",[]) or []
    mapping,node_matches=tolerant_match_nodes(pn,gn,w,h); ntp=len(node_matches); nfp=len(pn)-ntp; nfn=len(gn)-ntp; np_,nr,nf=prf(ntp,nfp,nfn)
    gt_pairs=[edge_pair(e) for e in ge]; pred_pairs=[mapped_edge_pair(e,mapping) for e in pe]; gc=Counter(gt_pairs); pc=Counter(pred_pairs)
    etp=counter_intersection(pc,gc); efp=len(pe)-etp; efn=len(ge)-etp; ep,er,ef=prf(etp,efp,efn)
    matched=pair_matched_edges(pe,ge,pred_pairs,gt_pairs); endpoint_values=[]
    for p,g in matched:
        for k in ("tail","head"):
            pp,gp=point_from_edge(p,k),point_from_edge(g,k)
            if pp is not None and gp is not None: endpoint_values.append(l2(pp,gp))
    node_ious=[m[2] for m in node_matches]
    gt_loop=Counter(p for p in gt_pairs if p[0] is not None and p[0]==p[1]); pred_loop=Counter(p for p in pred_pairs if p[0] is not None and p[0]==p[1]); matched_loop=counter_intersection(pred_loop,gt_loop)
    outdeg=defaultdict(int)
    for t,_h in gt_pairs:
        if t is not None: outdeg[t]+=1
    branch_sources={k for k,v in outdeg.items() if v>=2}; branch_gt=Counter(p for p in gt_pairs if p[0] in branch_sources); branch_match=counter_intersection(pc,branch_gt)
    edit=nfp+nfn+efp+efn; denom=max(1,max(len(pn),len(gn))+max(len(pe),len(ge)))
    gt_valid_endpoints=sum(int(point_from_edge(e,"tail") is not None)+int(point_from_edge(e,"head") is not None) for e in ge)
    return {"gt_nodes":len(gn),"pred_nodes":len(pn),"node_tp":ntp,"node_fp":nfp,"node_fn":nfn,"node_precision":np_,"node_recall":nr,"node_f1":nf,
            "matched_box_iou":float(np.mean(node_ious)) if node_ious else 0.,"matched_box_iou_values":node_ious,"gt_edges":len(ge),"pred_edges":len(pe),
            "edge_tp":etp,"edge_fp":efp,"edge_fn":efn,"edge_precision":ep,"edge_recall":er,"edge_f1":ef,"link_precision":ep,"link_recall":er,"link_f1":ef,
            "matched_directed_links":len(matched),"matched_link_coverage":safe_div(len(matched),len(ge)),"endpoint_l2_values":endpoint_values,
            "endpoint_l2":float(np.mean(endpoint_values)) if endpoint_values else float("nan"),"evaluated_endpoints":len(endpoint_values),"gt_valid_endpoints":gt_valid_endpoints,
            "endpoint_coverage":safe_div(len(endpoint_values),gt_valid_endpoints),"gt_self_loops":int(sum(gt_loop.values())),"matched_self_loops":matched_loop,
            "loop_back_recall":safe_div(matched_loop,sum(gt_loop.values())),"gt_branch_edges":int(sum(branch_gt.values())),"matched_branch_edges":branch_match,
            "decision_branch_recall":safe_div(branch_match,sum(branch_gt.values())),"nsee":safe_div(edit,denom),"structural_edit_cost":edit}

def strict_tdrr(pred,gt):
    pn,gn=pred.get("nodes",[]) or [],gt.get("nodes",[]) or []; pe,ge=pred.get("arrows",[]) or [],gt.get("arrows",[]) or []; cand=[]
    for pi,p in enumerate(pn):
        for gi,g in enumerate(gn):
            iou=bbox_iou_xywh(p["bbox"],g["bbox"])
            if iou>=TDRR_NODE_IOU_THRESHOLD: cand.append((-iou,pi,gi))
    cand.sort(); up=set(); ug=set(); mapping={}
    for _,pi,gi in cand:
        if pi in up or gi in ug: continue
        up.add(pi); ug.add(gi); mapping[str(pn[pi]["id"])]=str(gn[gi]["id"])
    exact_nodes=len(pn)==len(gn)==len(mapping)
    return int(exact_nodes and Counter(mapped_edge_pair(e,mapping) for e in pe)==Counter(edge_pair(e) for e in ge))

def flatten(rows,key):
    vals=[]
    for r in rows:
        for v in r.get(key,[]) or []:
            if math.isfinite(float(v)): vals.append(float(v))
    return vals

def aggregate_common(rows,group_key=None):
    groups=defaultdict(list)
    if group_key is None: groups["overall"]=rows
    else:
        for r in rows: groups[r[group_key]].append(r)
    result=[]
    for name,items in sorted(groups.items()):
        s=lambda k:sum(float(x.get(k,0) or 0) for x in items); np_,nr,nf=prf(int(s("node_tp")),int(s("node_fp")),int(s("node_fn"))); ep,er,ef=prf(int(s("edge_tp")),int(s("edge_fp")),int(s("edge_fn")))
        boxvals=flatten(items,"matched_box_iou_values"); endvals=flatten(items,"endpoint_l2_values")
        result.append({"group":name,"num_images":len(items),"node_precision":np_,"node_recall":nr,"node_f1":nf,
                       "mBox_IoU_macro":float(np.mean([x["matched_box_iou"] for x in items])),"mBox_IoU_pooled":float(np.mean(boxvals)) if boxvals else float("nan"),
                       "link_precision":ep,"link_recall":er,"link_f1":ef,"nSEE":float(np.mean([x["nsee"] for x in items])),
                       "endpoint_l2_mean":float(np.mean(endvals)) if endvals else float("nan"),"endpoint_l2_macro":float(np.mean([float(x["endpoint_l2"]) for x in items if math.isfinite(float(x.get("endpoint_l2", float("nan"))))])) if any(math.isfinite(float(x.get("endpoint_l2", float("nan")))) for x in items) else float("nan"),"endpoint_l2_median":float(np.median(endvals)) if endvals else float("nan"),
                       "endpoint_l2_p90":float(np.percentile(endvals,90)) if endvals else float("nan"),"endpoint_coverage":safe_div(len(endvals),s("gt_valid_endpoints")),
                       "loop_recall":safe_div(s("matched_self_loops"),s("gt_self_loops")),"branch_recall":safe_div(s("matched_branch_edges"),s("gt_branch_edges")),
                       "tdrr":float(np.mean([x.get("tdrr",0) for x in items])) if COMPUTE_TDRR else float("nan")})
    return result

In [ ]:
if TRAIN_PROFILE == "paper_90k":
    PROFILE_MAX_STEPS=90_000
    LR_MILESTONES=(60_000,80_000)
elif TRAIN_PROFILE == "strong_30k":
    PROFILE_MAX_STEPS=30_000
    LR_MILESTONES=(20_000,26_667)
else:
    raise ValueError("TRAIN_PROFILE must be 'strong_30k' or 'paper_90k'")

MAX_OPTIMIZER_STEPS=(
    int(MAX_OPTIMIZER_STEPS_OVERRIDE)
    if MAX_OPTIMIZER_STEPS_OVERRIDE is not None
    else PROFILE_MAX_STEPS
)
if QUICK_TEST:
    MAX_OPTIMIZER_STEPS=QUICK_TEST_STEPS
    LR_MILESTONES=(max(1,QUICK_TEST_STEPS//2),max(2,int(.8*QUICK_TEST_STEPS)))

MODE=RUN_MODE.lower().strip()
if MODE=="auto":
    MODE="evaluate_existing" if RESOLVED_BASELINE_CHECKPOINT is not None else "train"
if MODE not in {"train","evaluate_existing"}:
    raise ValueError("RUN_MODE must be 'auto', 'train', or 'evaluate_existing'")
if MODE=="evaluate_existing" and RESOLVED_BASELINE_CHECKPOINT is None:
    raise FileNotFoundError(
        "Evaluation-only mode requires BASELINE_CHECKPOINT to resolve to a .pt file."
    )

print("="*72)
print("RESOLVED RUN MODE:",MODE)
if MODE=="evaluate_existing":
    print("TRAINING WILL BE SKIPPED.")
    print("Checkpoint:",RESOLVED_BASELINE_CHECKPOINT)
else:
    print("Training profile:",TRAIN_PROFILE)
    print("Target optimizer steps:",MAX_OPTIMIZER_STEPS)
print("="*72)

DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Building Arrow R-CNN on",DEVICE)

# Existing checkpoints already contain the complete backbone.
pretrained_for_build=bool(USE_IMAGENET_PRETRAINED_BACKBONE and MODE=="train")
MODEL=build_arrow_rcnn(pretrained=pretrained_for_build)
MODEL.to(DEVICE)

GPU_COUNT=torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_MULTI_GPU=bool(
    MODE=="train"
    and DEVICE.type=="cuda"
    and USE_ALL_AVAILABLE_GPUS
    and GPU_COUNT>1
    and MULTI_GPU_STRATEGY=="detection_data_parallel"
)

def _move_nested_to_device(obj,device):
    if torch.is_tensor(obj):
        return obj.to(device,non_blocking=True)
    if isinstance(obj,dict):
        return {k:_move_nested_to_device(v,device) for k,v in obj.items()}
    if isinstance(obj,list):
        return [_move_nested_to_device(v,device) for v in obj]
    if isinstance(obj,tuple):
        return tuple(_move_nested_to_device(v,device) for v in obj)
    return obj

def _balanced_chunk_sizes(n,parts):
    parts=max(1,min(int(parts),int(n)))
    q,r=divmod(int(n),parts)
    return [q+(1 if i<r else 0) for i in range(parts)]

def _partition_detection_batch(images,targets,parts):
    """Pure-Python list partition used by the CUDA scatter path and unit tests."""
    images=list(images)
    targets=None if targets is None else list(targets)
    if targets is not None and len(images)!=len(targets):
        raise RuntimeError(
            f"Image/target batch mismatch: {len(images)} images vs {len(targets)} targets."
        )
    if not images:
        raise RuntimeError("Empty detection batch.")

    sizes=_balanced_chunk_sizes(len(images),parts)
    chunks=[]
    start=0
    for n in sizes:
        stop=start+n
        chunks.append(
            (images[start:stop], None if targets is None else targets[start:stop])
        )
        start=stop
    return sizes,chunks

def _weighted_reduce_loss_dicts(outputs,sizes,output_device):
    """Reduce per-replica scalar loss dictionaries while preserving autograd."""
    if not outputs:
        raise RuntimeError("No replica outputs to reduce.")
    keys=list(outputs[0].keys())
    if any(set(o.keys())!=set(keys) for o in outputs):
        raise RuntimeError("Replica loss dictionaries have inconsistent keys.")
    if len(outputs)!=len(sizes):
        raise RuntimeError(
            f"Replica output/count mismatch: outputs={len(outputs)}, sizes={len(sizes)}"
        )

    if isinstance(output_device,int):
        out_dev=torch.device(f"cuda:{output_device}")
    else:
        out_dev=torch.device(output_device)

    total=float(sum(sizes))
    gathered={}
    for key in keys:
        vals=[o[key].to(out_dev) for o in outputs]
        if any(v.ndim!=0 for v in vals):
            raise RuntimeError(
                f"Expected scalar loss '{key}' from each replica; "
                f"got {[tuple(v.shape) for v in vals]}."
            )
        gathered[key]=sum(
            v*(float(sz)/total)
            for v,sz in zip(vals,sizes)
        )
    return gathered

class DetectionDataParallel(nn.DataParallel):
    """
    DataParallel specialized for torchvision detection models.

    Vanilla nn.DataParallel recursively scatters every Tensor contained in the
    Python list of images. For a [C,H,W] image Tensor that means splitting C
    across GPUs, which is invalid for Faster R-CNN and caused the previous
    2-channel-vs-3-channel failure.

    This wrapper instead chunks the *list of images* across GPUs. Every replica
    receives whole [3,H,W] images and the corresponding target dictionaries.
    """
    def __init__(self,module,device_ids=None,output_device=None):
        super().__init__(module,device_ids=device_ids,output_device=output_device)
        self._last_chunk_sizes=[]
        self._printed_scatter_audit=False

    def scatter(self,inputs,kwargs,device_ids):
        if not inputs:
            return super().scatter(inputs,kwargs,device_ids)

        images=inputs[0]
        targets=inputs[1] if len(inputs)>1 else None
        remaining=inputs[2:]

        if not isinstance(images,(list,tuple)):
            raise TypeError(
                "DetectionDataParallel expects the first positional argument "
                "to be a list/tuple of image tensors."
            )
        if targets is not None and not isinstance(targets,(list,tuple)):
            raise TypeError(
                "DetectionDataParallel expects targets to be a list/tuple or None."
            )
        if targets is not None and len(images)!=len(targets):
            raise RuntimeError(
                f"Image/target batch mismatch: {len(images)} images vs {len(targets)} targets."
            )
        if len(images)==0:
            raise RuntimeError("Empty image batch is not valid for Faster R-CNN.")

        nrep=min(len(device_ids),len(images))
        chunk_sizes,chunks=_partition_detection_batch(images,targets,nrep)
        self._last_chunk_sizes=chunk_sizes

        scattered_inputs=[]
        scattered_kwargs=[]

        for replica_idx,(img_source,tgt_source) in enumerate(chunks):
            dev_index=int(device_ids[replica_idx])
            dev=torch.device(f"cuda:{dev_index}")

            img_chunk=[]
            for img in img_source:
                if not torch.is_tensor(img) or img.ndim!=3:
                    raise RuntimeError(
                        f"Expected each image to be a [C,H,W] Tensor, got {type(img)} "
                        f"with shape {getattr(img,'shape',None)}."
                    )
                if REQUIRE_THREE_CHANNEL_IMAGES and int(img.shape[0])!=3:
                    raise RuntimeError(
                        f"Invalid image channels before GPU scatter: shape={tuple(img.shape)}. "
                        "Expected 3 channels."
                    )
                img_chunk.append(img.to(dev,non_blocking=True))

            if tgt_source is None:
                tgt_chunk=None
            else:
                tgt_chunk=[
                    _move_nested_to_device(t,dev)
                    for t in tgt_source
                ]

            extra=tuple(_move_nested_to_device(x,dev) for x in remaining)
            if targets is None:
                replica_args=(img_chunk,*extra)
            else:
                replica_args=(img_chunk,tgt_chunk,*extra)

            replica_kwargs=_move_nested_to_device(dict(kwargs),dev)
            scattered_inputs.append(replica_args)
            scattered_kwargs.append(replica_kwargs)

        if not self._printed_scatter_audit:
            channel_layout=[
                [int(img.shape[0]) for img in args[0]]
                for args in scattered_inputs
            ]
            print(
                "[multi-gpu scatter audit] "
                f"chunk_sizes={chunk_sizes} | channels_per_replica={channel_layout}"
            )
            if REQUIRE_THREE_CHANNEL_IMAGES and any(
                c!=3 for replica in channel_layout for c in replica
            ):
                raise RuntimeError("Multi-GPU scatter audit found a non-3-channel image.")
            self._printed_scatter_audit=True

        return tuple(scattered_inputs),tuple(scattered_kwargs)

    def gather(self,outputs,output_device):
        # During training GeneralizedRCNN returns one dictionary of scalar losses
        # from each replica. Reduce them to one scalar-loss dictionary on cuda:0.
        if outputs and all(isinstance(o,dict) for o in outputs):
            keys=list(outputs[0].keys())
            if any(set(o.keys())!=set(keys) for o in outputs):
                raise RuntimeError("Replica loss dictionaries have inconsistent keys.")

            sizes=self._last_chunk_sizes or [1]*len(outputs)
            return _weighted_reduce_loss_dicts(outputs,sizes,output_device)

        return super().gather(outputs,output_device)

if USE_MULTI_GPU:
    TRAIN_MODEL=DetectionDataParallel(
        MODEL,
        device_ids=list(range(GPU_COUNT)),
        output_device=0,
    )
else:
    TRAIN_MODEL=MODEL

print("GPU count available :",GPU_COUNT)
print("Multi-GPU training  :",USE_MULTI_GPU)
if USE_MULTI_GPU:
    print("Training wrapper    : DetectionDataParallel")
    print("GPU device IDs      :",list(range(GPU_COUNT)))
    print("Inference/evaluation: unwrapped MODEL on cuda:0")
else:
    print("Training wrapper    : single-device")

# CPU-only unit tests for the exact list partition and loss reduction used by
# the CUDA wrapper. These explicitly guard against the old channel-splitting bug.
assert _balanced_chunk_sizes(4,2)==[2,2]
assert _balanced_chunk_sizes(3,2)==[2,1]
assert _balanced_chunk_sizes(1,2)==[1]

_fake_images=[torch.zeros(3,17+i,19+i) for i in range(4)]
_fake_targets=[
    {"boxes":torch.tensor([[1.,1.,5.,5.]]),"labels":torch.tensor([1])}
    for _ in range(4)
]
_sizes,_chunks=_partition_detection_batch(_fake_images,_fake_targets,2)
assert _sizes==[2,2]
assert [len(c[0]) for c in _chunks]==[2,2]
assert [len(c[1]) for c in _chunks]==[2,2]
assert all(int(img.shape[0])==3 for c in _chunks for img in c[0])

_a=torch.tensor(2.0,requires_grad=True)
_b=torch.tensor(4.0,requires_grad=True)
_reduced=_weighted_reduce_loss_dicts(
    [{"loss":_a},{"loss":_b}],[2,2],torch.device("cpu")
)["loss"]
assert _reduced.ndim==0 and torch.allclose(_reduced,torch.tensor(3.0))
_reduced.backward()
assert torch.allclose(_a.grad,torch.tensor(0.5))
assert torch.allclose(_b.grad,torch.tensor(0.5))

print("Detection-aware split + scalar-loss reduction regression tests: PASS")

def load_checkpoint_into_model(model,path,load_optimizer=False,optimizer=None,scheduler=None):
    ckpt=torch.load(path,map_location="cpu",weights_only=False)
    state=ckpt["model"] if isinstance(ckpt,dict) and "model" in ckpt else ckpt

    target_model=unwrap_model(model)
    missing,unexpected=target_model.load_state_dict(state,strict=False)
    if missing or unexpected:
        raise RuntimeError(
            "Checkpoint/model state mismatch. Refusing partial loading.\n"
            f"Missing keys: {missing}\nUnexpected keys: {unexpected}"
        )

    print("Checkpoint loaded exactly:",path)
    step=int(ckpt.get("step",0)) if isinstance(ckpt,dict) else 0
    if isinstance(ckpt,dict):
        print("Checkpoint step:",step)
        if "train_profile" in ckpt:
            print("Checkpoint train_profile:",ckpt["train_profile"])
        if ckpt.get("config"):
            print("Checkpoint config metadata:",ckpt["config"])

    if load_optimizer and isinstance(ckpt,dict):
        if optimizer is not None and "optimizer" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scheduler is not None and "scheduler" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler"])
    return step

if MODE=="evaluate_existing":
    LOADED_CHECKPOINT_STEP=load_checkpoint_into_model(MODEL,RESOLVED_BASELINE_CHECKPOINT)
    MODEL.eval()


## 5. Training

Optimizer settings follow the paper: SGD, momentum 0.9, weight decay 1e-4,
and base LR 0.005 for effective batch size 4. AMP and gradient accumulation
are engineering adaptations for current hosted-runtime T4 runtimes.

In [ ]:
# ============================================================
# END-TO-END PRETRAINING PREFLIGHT
# Runs inference -> graph conversion -> common evaluator BEFORE
# any 30k training can consume GPU hours.
# ============================================================
@torch.inference_mode()
def pretraining_pipeline_preflight():
    if len(VAL_DATASET)==0:
        raise RuntimeError("Validation dataset is empty.")
    MODEL.eval()
    image,_target,sample=VAL_DATASET[0]
    out=MODEL([image.to(DEVICE)])[0]
    required={"boxes","scores","labels","keypoints"}
    missing=required-set(out.keys())
    if missing:
        raise RuntimeError(f"Model inference output is missing fields: {sorted(missing)}")

    raw={k:v.detach().cpu() for k,v in out.items() if k in required}
    n=len(raw["boxes"])
    assert raw["scores"].shape[0]==n
    assert raw["labels"].shape[0]==n
    assert raw["keypoints"].shape==(n,2,3)

    pred=graph_from_detection(raw,sample,DEFAULT_SCORE_THRESHOLD)
    gt=load_gt_graph(sample)
    metrics=evaluate_common(pred,gt)
    if COMPUTE_TDRR:
        _=strict_tdrr(pred,gt)

    # Ensure final graph JSON is serializable.
    json.dumps(pred,allow_nan=False)
    print(
        "End-to-end pretraining preflight: PASS | "
        f"detections={n}, pred_nodes={len(pred.get('nodes',[]))}, "
        f"pred_edges={len(pred.get('arrows',[]))}, "
        f"sanity_link_f1={metrics['link_f1']:.4f}"
    )
    MODEL.train() if MODE=="train" else MODEL.eval()

pretraining_pipeline_preflight()


In [ ]:
TRAIN_HISTORY=[]

def current_lr(optimizer): return float(optimizer.param_groups[0]["lr"])

def save_training_checkpoint(path,model,optimizer,scheduler,step,physical_bs,accum_steps):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    torch.save({
        "model":unwrap_model(model).state_dict(),"optimizer":optimizer.state_dict(),"scheduler":scheduler.state_dict(),
        "step":int(step),"physical_batch_size":int(physical_bs),"gradient_accumulation_steps":int(accum_steps),
        "config":{"train_profile":TRAIN_PROFILE,"max_optimizer_steps":MAX_OPTIMIZER_STEPS,"base_lr_batch4":BASE_LR_FOR_BATCH4,
                  "momentum":MOMENTUM,"weight_decay":WEIGHT_DECAY,"arrow_proposal_iou_threshold":ARROW_PROPOSAL_IOU_THRESHOLD,
                  "arrow_loss_weight":ARROW_LOSS_WEIGHT,"quick_test":bool(QUICK_TEST),
                  "lr_milestones":list(LR_MILESTONES),
                  "arrow_endpoint_convention":ARROW_ENDPOINT_CONVENTION,
                  "common_canvas":[COMMON_CANVAS_WIDTH,COMMON_CANVAS_HEIGHT],
                  "auto_resize_images_to_json_canvas":bool(AUTO_RESIZE_IMAGES_TO_JSON_CANVAS),
                  "use_multi_gpu":bool(USE_MULTI_GPU),
                  "gpu_count":int(GPU_COUNT),
                  "multi_gpu_strategy":MULTI_GPU_STRATEGY}
    },path)

def prepare_training_batch(images,targets):
    """
    Multi-GPU wrapper receives CPU/pinned-memory lists and performs its own
    list-level scatter. Single-GPU mode moves the batch to cuda:0 here.
    """
    images=list(images)
    targets=list(targets)

    for img in images:
        if img.ndim!=3:
            raise RuntimeError(f"Invalid training image shape: {tuple(img.shape)}")
        if REQUIRE_THREE_CHANNEL_IMAGES and int(img.shape[0])!=3:
            raise RuntimeError(
                f"Training image has {int(img.shape[0])} channels; expected 3."
            )

    if USE_MULTI_GPU:
        return images,targets

    images=[x.to(DEVICE,non_blocking=True) for x in images]
    targets=[
        {k:(v.to(DEVICE,non_blocking=True) if torch.is_tensor(v) else v)
         for k,v in t.items()}
        for t in targets
    ]
    return images,targets

def choose_batch_size(model):
    """
    Real forward+backward probe before long training.

    In dual-GPU mode this exercises the exact DetectionDataParallel scatter,
    replica forward, loss reduction, and backward path that the 90k run uses.
    """
    requested=max(1,int(PHYSICAL_BATCH_SIZE))
    candidates=[requested]
    if AUTO_REDUCE_BATCH_ON_OOM and requested>1:
        candidates.extend(range(requested-1,0,-1))

    last_non_oom_error=None

    for bs in candidates:
        loader=make_loader(TRAIN_DATASET,bs,True)
        images,targets,_=next(iter(loader))
        images,targets=prepare_training_batch(images,targets)

        try:
            model.train()
            model.zero_grad(set_to_none=True)
            probe_scaler=torch.amp.GradScaler(
                "cuda",enabled=(USE_AMP and DEVICE.type=="cuda")
            )
            with torch.amp.autocast(
                "cuda",enabled=(USE_AMP and DEVICE.type=="cuda")
            ):
                losses=model(images,targets)

                required_loss_keys={
                    "loss_classifier","loss_box_reg","loss_arrow_reg",
                    "loss_objectness","loss_rpn_box_reg"
                }
                missing=required_loss_keys-set(losses)
                if missing:
                    raise RuntimeError(
                        f"Training forward is missing loss terms: {sorted(missing)}"
                    )
                if any(v.ndim!=0 for v in losses.values()):
                    raise RuntimeError(
                        "Training loss reduction failed; expected scalar losses, got "
                        f"{ {k:tuple(v.shape) for k,v in losses.items()} }"
                    )

                loss=sum(losses.values())

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite probe loss: {float(loss.detach().cpu())}")

            probe_scaler.scale(loss).backward()

            # Confirm the base model actually received gradients after replica backward.
            grad_count=sum(
                1 for p in unwrap_model(model).parameters()
                if p.requires_grad and p.grad is not None
            )
            if grad_count==0:
                raise RuntimeError(
                    "Multi-GPU backward completed but no gradients reached the base model."
                )

            model.zero_grad(set_to_none=True)
            del losses,loss,images,targets,loader,probe_scaler
            clear_cuda_cache_all()

            per_gpu=(
                _balanced_chunk_sizes(bs,GPU_COUNT)
                if USE_MULTI_GPU else [bs]
            )
            print(
                "Training forward+backward preflight: PASS | "
                f"global_batch={bs} | per_gpu_chunks={per_gpu} | "
                f"base_params_with_grad={grad_count}"
            )
            return bs

        except torch.cuda.OutOfMemoryError:
            print("OOM at global physical batch",bs,"during forward/backward probe")
            model.zero_grad(set_to_none=True)
            clear_cuda_cache_all()
            continue
        except Exception as exc:
            # A programming/shape/parallelism error must never be disguised as OOM.
            last_non_oom_error=exc
            model.zero_grad(set_to_none=True)
            clear_cuda_cache_all()
            raise

    if last_non_oom_error is not None:
        raise last_non_oom_error
    raise RuntimeError(
        "Even global physical batch size 1 OOMed during the forward/backward probe."
    )

def memory_guard_or_checkpoint(step,model,optimizer,scheduler,physical_bs,accum_steps,checkpoint_path):
    """
    Prevent a silent OS-level kernel kill when host RAM gets critically low.
    Save a resumable checkpoint first, then stop cleanly.
    """
    if not MEMORY_GUARD_ENABLED:
        return

    m=print_host_memory(f"step {step}")
    avail=m["available_gb"]
    if math.isfinite(avail) and avail < float(MIN_AVAILABLE_HOST_RAM_GB):
        print(
            f"[ram-guard] Only {avail:.2f} GB host RAM available. "
            "Saving emergency checkpoint before stopping."
        )
        save_training_checkpoint(
            checkpoint_path,model,optimizer,scheduler,
            step,physical_bs,accum_steps
        )
        gc.collect()
        clear_cuda_cache_all()
        raise RuntimeError(
            "RAM guard stopped training before an OS-level kernel kill. "
            f"Resume from {checkpoint_path}."
        )

if MODE=="train":
    physical_bs=choose_batch_size(TRAIN_MODEL)
    accum_steps=max(1,int(math.ceil(EFFECTIVE_BATCH_SIZE/physical_bs)))
    effective_bs=physical_bs*accum_steps
    base_lr=BASE_LR_FOR_BATCH4*(effective_bs/4.0)

    chunks=(
        _balanced_chunk_sizes(physical_bs,GPU_COUNT)
        if USE_MULTI_GPU else [physical_bs]
    )
    print(
        f"global physical batch={physical_bs}, per-GPU chunks={chunks}, "
        f"accumulation={accum_steps}, effective batch={effective_bs}, lr={base_lr}"
    )

    TRAIN_LOADER=make_loader(TRAIN_DATASET,physical_bs,True)
    optimizer=torch.optim.SGD(
        [p for p in unwrap_model(TRAIN_MODEL).parameters() if p.requires_grad],
        lr=base_lr,momentum=MOMENTUM,weight_decay=WEIGHT_DECAY
    )
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=list(LR_MILESTONES),gamma=.1)
    scaler=torch.amp.GradScaler("cuda",enabled=(USE_AMP and DEVICE.type=="cuda"))

    start_step=0
    local_last=OUT/"checkpoints"/"arrow_rcnn_common_last.pt"

    if RESUME_TRAINING:
        resume_candidate=(
            Path(RESOLVED_BASELINE_CHECKPOINT)
            if RESOLVED_BASELINE_CHECKPOINT is not None
            else local_last
        )
        if not resume_candidate.is_file():
            raise FileNotFoundError(
                "RESUME_TRAINING=True but no checkpoint was found. "
                "Set BASELINE_CHECKPOINT to a compatible arrow_rcnn_common_last.pt."
            )
        raw_ckpt=torch.load(resume_candidate,map_location="cpu",weights_only=False)
        cfgmeta=raw_ckpt.get("config",{}) if isinstance(raw_ckpt,dict) else {}
        old_quick=bool(cfgmeta.get("quick_test",False))
        if old_quick!=bool(QUICK_TEST):
            raise RuntimeError(
                f"Refusing incompatible resume: checkpoint quick_test={old_quick}, current QUICK_TEST={QUICK_TEST}."
            )
        old_profile=cfgmeta.get("train_profile")
        if old_profile is not None and old_profile!=TRAIN_PROFILE:
            raise RuntimeError(
                f"Refusing incompatible resume: checkpoint profile={old_profile}, current profile={TRAIN_PROFILE}."
            )
        start_step=load_checkpoint_into_model(
            TRAIN_MODEL,resume_candidate,True,optimizer,scheduler
        )
        print(f"Resuming training from optimizer step {start_step}")
    else:
        print("Fresh training requested; no checkpoint is auto-resumed.")

    print_host_memory("before long training loop")
    TRAIN_MODEL.train(); optimizer.zero_grad(set_to_none=True)
    opt_step=start_step; micro_step=0; rolling=defaultdict(float); rolling_n=0; t0=time.time()

    while opt_step < MAX_OPTIMIZER_STEPS:
        for images,targets,_samples in TRAIN_LOADER:
            images,targets=prepare_training_batch(images,targets)
            with torch.amp.autocast("cuda",enabled=(USE_AMP and DEVICE.type=="cuda")):
                loss_dict=TRAIN_MODEL(images,targets)
                if any(v.ndim!=0 for v in loss_dict.values()):
                    raise RuntimeError(
                        "Expected scalar training losses, got "
                        f"{ {k:tuple(v.shape) for k,v in loss_dict.items()} }"
                    )
                total=sum(loss_dict.values())
                if not torch.isfinite(total):
                    raise RuntimeError(
                        f"Non-finite training loss at optimizer step {opt_step}: "
                        f"{float(total.detach().cpu())}"
                    )
                scaled_loss=total/accum_steps
            scaler.scale(scaled_loss).backward(); micro_step+=1
            for k,v in loss_dict.items():
                rolling[k]+=float(v.detach().cpu())
            rolling["loss_total"]+=float(total.detach().cpu()); rolling_n+=1

            # The autograd graph is no longer needed after backward. Releasing
            # these references immediately avoids retaining a full detection
            # batch until the next iteration.
            del loss_dict,total,scaled_loss,images,targets

            if micro_step % accum_steps != 0:
                if micro_step % max(1,GC_EVERY_STEPS*accum_steps)==0:
                    gc.collect()
                continue
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step(); opt_step+=1

            if opt_step % max(1,int(GC_EVERY_STEPS))==0:
                gc.collect()

            if opt_step % max(1,int(MEMORY_CHECK_EVERY_STEPS))==0:
                memory_guard_or_checkpoint(
                    opt_step,TRAIN_MODEL,optimizer,scheduler,
                    physical_bs,accum_steps,local_last
                )

            if opt_step<=3 or opt_step%LOG_EVERY_STEPS==0:
                row={"step":opt_step,"lr":current_lr(optimizer),"elapsed_min":(time.time()-t0)/60.0}
                for k,v in rolling.items(): row[k]=v/max(1,rolling_n)
                TRAIN_HISTORY.append(row)
                print(f"[train] {opt_step}/{MAX_OPTIMIZER_STEPS} loss={row.get('loss_total',float('nan')):.4f} "
                      f"arrow={row.get('loss_arrow_reg',float('nan')):.4f} lr={row['lr']:.2e} elapsed={row['elapsed_min']:.1f}m")
                rolling=defaultdict(float); rolling_n=0

            if opt_step == min(5, MAX_OPTIMIZER_STEPS):
                elapsed_s=max(time.time()-t0,1e-6)
                sec_per_step=elapsed_s/max(opt_step-start_step,1)
                remaining_steps=max(MAX_OPTIMIZER_STEPS-opt_step,0)
                eta_h=remaining_steps*sec_per_step/3600.0
                full30_h=30000*sec_per_step/3600.0
                full90_h=90000*sec_per_step/3600.0
                print(
                    f"[speed] {sec_per_step:.3f} s/optimizer-step | "
                    f"ETA current run={eta_h:.2f} h | "
                    f"30k≈{full30_h:.2f} h | 90k≈{full90_h:.2f} h"
                )

            if opt_step%CHECKPOINT_EVERY_STEPS==0:
                save_training_checkpoint(
                    local_last,TRAIN_MODEL,optimizer,scheduler,
                    opt_step,physical_bs,accum_steps
                )
                gc.collect()
                clear_cuda_cache_all()
                print_host_memory(f"after checkpoint {opt_step}")
            if opt_step>=MAX_OPTIMIZER_STEPS: break

    save_training_checkpoint(local_last,TRAIN_MODEL,optimizer,scheduler,opt_step,physical_bs,accum_steps)
    final_path=OUT/"checkpoints"/"arrow_rcnn_common_final.pt"
    torch.save({
        "model":unwrap_model(TRAIN_MODEL).state_dict(),
        "step":opt_step,
        "train_profile":TRAIN_PROFILE,
        "physical_batch_size":physical_bs,
        "effective_batch_size":effective_bs,
        "config":{
            "arrow_endpoint_convention":ARROW_ENDPOINT_CONVENTION,
            "lr_milestones":list(LR_MILESTONES),
            "common_canvas":[COMMON_CANVAS_WIDTH,COMMON_CANVAS_HEIGHT],
            "auto_resize_images_to_json_canvas":bool(AUTO_RESIZE_IMAGES_TO_JSON_CANVAS),
            "use_multi_gpu":bool(USE_MULTI_GPU),
            "gpu_count":int(GPU_COUNT),
            "multi_gpu_strategy":MULTI_GPU_STRATEGY,
            "global_physical_batch_size":int(physical_bs),
            "effective_batch_size":int(effective_bs),
        }
    },final_path)

    if TRAIN_HISTORY:
        keys=sorted({k for r in TRAIN_HISTORY for k in r})
        with open(OUT/"training_history.csv","w",newline="",encoding="utf-8") as f:
            w=csv.DictWriter(f,fieldnames=keys); w.writeheader(); w.writerows(TRAIN_HISTORY)
    print("Training complete:",final_path)

MODEL.eval()

## 6. Validation-set threshold selection

The detector is run once at a permissive internal threshold. The final score
threshold is selected **only on the common validation split** by Link F1, with
Node F1 and nSEE as tie-breakers. The selected threshold is then frozen for the
held-out test set.

In [ ]:
def cpu_detection(out): return {k:v.detach().cpu() for k,v in out.items() if k in {"boxes","scores","labels","keypoints"}}

@torch.inference_mode()
def infer_raw(model,dataset,label="infer"):
    model.eval(); loader=make_loader(dataset,1,False); raws=[]
    for i,(images,_targets,_samples) in enumerate(loader,1):
        image=images[0].to(DEVICE,non_blocking=True); out=model([image])[0]; raws.append(cpu_detection(out))
        if i<=3 or i%25==0: print(f"[{label}] {i}/{len(dataset)}")
    return raws

def evaluate_raw_at_threshold(raws,records,threshold):
    rows=[]
    for raw,record in zip(raws,records):
        sample=record["sample"]; pred=graph_from_detection(raw,sample,threshold); gt=load_gt_graph(sample); row=evaluate_common(pred,gt)
        row.update({"family":sample.family,"sample_id":sample.sample_id})
        if COMPUTE_TDRR: row["tdrr"]=strict_tdrr(pred,gt)
        rows.append(row)
    return aggregate_common(rows)[0],rows

print("Running common-validation inference...")
VAL_RAWS=infer_raw(MODEL,VAL_DATASET,"val")
threshold_rows=[]
if AUTO_TUNE_SCORE_THRESHOLD:
    for t in SCORE_THRESHOLD_CANDIDATES:
        agg,_=evaluate_raw_at_threshold(VAL_RAWS,VAL_RECORDS,float(t))
        threshold_rows.append({"score_threshold":float(t),"link_f1":agg["link_f1"],"node_f1":agg["node_f1"],"nSEE":agg["nSEE"],"endpoint_l2_mean":agg["endpoint_l2_mean"]})
        print(f"threshold={t:.2f} | Link F1={agg['link_f1']:.4f} | Node F1={agg['node_f1']:.4f} | nSEE={agg['nSEE']:.4f}")
    threshold_rows.sort(key=lambda r:(r["link_f1"],r["node_f1"],-r["nSEE"]),reverse=True)
    SELECTED_SCORE_THRESHOLD=float(threshold_rows[0]["score_threshold"])
else:
    SELECTED_SCORE_THRESHOLD=float(DEFAULT_SCORE_THRESHOLD)
print("Selected score threshold:",SELECTED_SCORE_THRESHOLD)

if threshold_rows:
    with open(OUT/"validation_threshold_sweep.csv","w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=list(threshold_rows[0])); w.writeheader(); w.writerows(threshold_rows)
with open(OUT/"selected_inference_config.json","w",encoding="utf-8") as f:
    json.dump({"score_threshold":SELECTED_SCORE_THRESHOLD,"node_nms_iou":NODE_NMS_IOU,"arrow_nms_iou":ARROW_NMS_IOU,
               "text_nms_iou":TEXT_NMS_IOU,"fa_initial_arrow_tail_distance_px":FA_INITIAL_ARROW_TAIL_DISTANCE_PX},f,indent=2)

def evaluate_reversed_direction_at_threshold(raws,records,threshold):
    rows=[]
    for raw,record in zip(raws,records):
        sample=record["sample"]
        pred=graph_from_detection(raw,sample,threshold)
        pred=reverse_predicted_graph_direction(pred)
        gt=load_gt_graph(sample)
        row=evaluate_common(pred,gt)
        row.update({"family":sample.family,"sample_id":sample.sample_id})
        rows.append(row)
    return aggregate_common(rows)[0]

_corrected_val,_=evaluate_raw_at_threshold(
    VAL_RAWS,VAL_RECORDS,SELECTED_SCORE_THRESHOLD
)
_reversed_val=evaluate_reversed_direction_at_threshold(
    VAL_RAWS,VAL_RECORDS,SELECTED_SCORE_THRESHOLD
)

print("\n"+"="*72)
print("DIRECTION-SEMANTICS VALIDATION CHECK")
print("="*72)
print(
    f"Audited convention      : Link F1={_corrected_val['link_f1']:.4f} "
    f"| Endpoint L2={_corrected_val['endpoint_l2_mean']:.3f}"
)
print(
    f"Intentionally reversed  : Link F1={_reversed_val['link_f1']:.4f} "
    f"| Endpoint L2={_reversed_val['endpoint_l2_mean']:.3f}"
)

with open(OUT/"metrics"/"direction_semantics_check.json","w",encoding="utf-8") as f:
    json.dump({
        "endpoint_convention":ARROW_ENDPOINT_CONVENTION,
        "threshold":SELECTED_SCORE_THRESHOLD,
        "audited_convention":_corrected_val,
        "intentionally_reversed":_reversed_val,
    },f,indent=2,allow_nan=True)

if REPORT_FIXED_PAPER_THRESHOLD:
    _fixed_val,_=evaluate_raw_at_threshold(
        VAL_RAWS,VAL_RECORDS,float(PAPER_REFERENCE_SCORE_THRESHOLD)
    )
    print(
        f"Fixed threshold {PAPER_REFERENCE_SCORE_THRESHOLD:.2f}: "
        f"Link F1={_fixed_val['link_f1']:.4f} | "
        f"Node F1={_fixed_val['node_f1']:.4f} | nSEE={_fixed_val['nSEE']:.4f}"
    )


## 7. Held-out 450-image test inference and common-protocol metrics

In [ ]:
TEST_RAWS=infer_raw(MODEL,TEST_DATASET,"test")
PRED_ROOT=OUT/"predictions"
if SAVE_GRAPH_PREDICTIONS: PRED_ROOT.mkdir(parents=True,exist_ok=True)

metric_rows=[]
for raw,record in zip(TEST_RAWS,TEST_RECORDS):
    sample=record["sample"]; pred=graph_from_detection(raw,sample,SELECTED_SCORE_THRESHOLD); gt=load_gt_graph(sample)
    if SAVE_GRAPH_PREDICTIONS:
        p=PRED_ROOT/sample.family/sample.split/"json"/f"{sample.sample_id}.json"; p.parent.mkdir(parents=True,exist_ok=True)
        with open(p,"w",encoding="utf-8") as f: json.dump(pred,f,indent=2)
    row=evaluate_common(pred,gt); row.update({"family":sample.family,"split":sample.split,"sample_id":sample.sample_id,"image_name":sample.image_path.name})
    if COMPUTE_TDRR: row["tdrr"]=strict_tdrr(pred,gt)
    metric_rows.append(row)

overall=aggregate_common(metric_rows)[0]; by_family=aggregate_common(metric_rows,"family")
print("\n"+"="*72); print("ARROW R-CNN REIMPLEMENTATION — COMMON PROTOCOL"); print("="*72); print(json.dumps(overall,indent=2,allow_nan=True))
print("\nBY FAMILY"); print(json.dumps(by_family,indent=2,allow_nan=True))

def csv_safe(v): return json.dumps(v) if isinstance(v,(list,dict,tuple)) else v

def write_rows(path,rows):
    rows=[{k:csv_safe(v) for k,v in r.items()} for r in rows]; keys=[]
    for r in rows:
        for k in r:
            if k not in keys: keys.append(k)
    with open(path,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=keys); w.writeheader(); w.writerows(rows)

write_rows(OUT/"metrics"/"metrics_per_image.csv",metric_rows); write_rows(OUT/"metrics"/"metrics_by_family.csv",by_family); write_rows(OUT/"metrics"/"metrics_overall.csv",[overall])
summary={"method":"Arrow R-CNN reimplementation — topology-adapted common protocol","score_threshold":SELECTED_SCORE_THRESHOLD,"overall":overall,"by_family":by_family,
         "successful_images":len(metric_rows),"notes":{"Soft-clDice":"N/A: Arrow R-CNN predicts boxes and endpoints, not connector centerlines.",
         "node_semantics":"Collapsed to one generic node class for topology-only fairness.","text":"Trained as distractor class and excluded from graph evaluation.",
         "train_split":"Same family-stratified learning split as proposed framework (seed 39)."}}
with open(OUT/"metrics"/"metrics_summary.json","w",encoding="utf-8") as f: json.dump(summary,f,indent=2,allow_nan=True)

if COMPRESS_PREDICTIONS and PRED_ROOT.is_dir():
    zip_path=OUT/"arrow_rcnn_graph_predictions.zip"
    with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=9) as zf:
        for p in PRED_ROOT.rglob("*.json"): zf.write(p,arcname=str(Path("predictions")/p.relative_to(PRED_ROOT)).replace("\\","/"))
    print("Prediction archive:",zip_path,f"({zip_path.stat().st_size/1024**2:.2f} MB)")
    if DELETE_UNCOMPRESSED_AFTER_ZIP: shutil.rmtree(PRED_ROOT,ignore_errors=True)

if REPORT_FIXED_PAPER_THRESHOLD:
    fixed_rows=[]
    for raw,record in zip(TEST_RAWS,TEST_RECORDS):
        sample=record["sample"]
        pred_fixed=graph_from_detection(
            raw,sample,float(PAPER_REFERENCE_SCORE_THRESHOLD)
        )
        gt=load_gt_graph(sample)
        row=evaluate_common(pred_fixed,gt)
        row.update({
            "family":sample.family,
            "split":sample.split,
            "sample_id":sample.sample_id,
            "image_name":sample.image_path.name,
        })
        if COMPUTE_TDRR:
            row["tdrr"]=strict_tdrr(pred_fixed,gt)
        fixed_rows.append(row)

    fixed_overall=aggregate_common(fixed_rows)[0]
    fixed_by_family=aggregate_common(fixed_rows,"family")

    print("\n"+"="*72)
    print(f"FIXED PAPER-REFERENCE THRESHOLD = {PAPER_REFERENCE_SCORE_THRESHOLD:.2f}")
    print("="*72)
    print(json.dumps(fixed_overall,indent=2,allow_nan=True))

    write_rows(OUT/"metrics"/"metrics_overall_fixed_threshold_0p70.csv",[fixed_overall])
    write_rows(OUT/"metrics"/"metrics_by_family_fixed_threshold_0p70.csv",fixed_by_family)
    with open(OUT/"metrics"/"metrics_fixed_threshold_0p70.json","w",encoding="utf-8") as f:
        json.dump({
            "threshold":float(PAPER_REFERENCE_SCORE_THRESHOLD),
            "overall":fixed_overall,
            "by_family":fixed_by_family,
        },f,indent=2,allow_nan=True)


## 8. Direct comparison with the verified proposed-framework run

These proposed-framework values are frozen from the verified 450-image revision
run. If the proposed model changes, replace this block from its new
`metrics_summary.json` before using the comparison table in the manuscript.

In [ ]:
# Manuscript-reported proposed-framework reference.
# Primary-table fields use the same macro aggregation convention as the paper;
# pooled geometry diagnostics are listed separately.
PROPOSED_REFERENCE = {
    "overall": {
        "node_precision": 0.977587343441002,
        "node_recall": 0.9939678284182306,
        "node_f1": 0.9857095380525092,
        "mBox_IoU_macro": 0.8271,
        "link_precision": 0.901685393258427,
        "link_recall": 0.9494232475598935,
        "link_f1": 0.9249387696297363,
        "nSEE": 0.090,
        "endpoint_l2_macro": 3.77,
        "endpoint_coverage": 0.9494,
        "loop_recall": 0.9524,
        "branch_recall": 0.9404,
        "soft_cldice_macro": 0.9672,
        "soft_cldice_pooled": 0.9935,
        "endpoint_l2_pooled": 3.44,
    },
    "fa": {"link_f1":0.9202,"nSEE":0.096,"endpoint_l2_macro":3.54},
    "fca":{"link_f1":0.9238,"nSEE":0.103,"endpoint_l2_macro":5.00},
    "fcb":{"link_f1":0.9287,"nSEE":0.077,"endpoint_l2_macro":2.79},
}

comparison = [
    {
        "Method":"Arrow R-CNN common-protocol reimplementation",
        "Node F1":100*overall["node_f1"],
        "mBox-IoU":100*overall["mBox_IoU_macro"],
        "Link F1":100*overall["link_f1"],
        "nSEE":overall["nSEE"],
        "Endpoint L2":overall.get("endpoint_l2_macro", overall["endpoint_l2_mean"]),
        "Loop recall":100*overall["loop_recall"],
        "Branch recall":100*overall["branch_recall"],
    },
    {
        "Method":"Proposed framework",
        "Node F1":100*PROPOSED_REFERENCE["overall"]["node_f1"],
        "mBox-IoU":100*PROPOSED_REFERENCE["overall"]["mBox_IoU_macro"],
        "Link F1":100*PROPOSED_REFERENCE["overall"]["link_f1"],
        "nSEE":PROPOSED_REFERENCE["overall"]["nSEE"],
        "Endpoint L2":PROPOSED_REFERENCE["overall"]["endpoint_l2_macro"],
        "Loop recall":100*PROPOSED_REFERENCE["overall"]["loop_recall"],
        "Branch recall":100*PROPOSED_REFERENCE["overall"]["branch_recall"],
    },
]
print("\n"+"="*72)
print("COMMON-PROTOCOL COMPARISON — MANUSCRIPT PRIMARY METRICS")
print("="*72)
for r in comparison:
    print(json.dumps(r, indent=2))
write_rows(OUT/"metrics"/"comparison_with_proposed.csv", comparison)

pooled_geometry_diagnostics = {
    "Proposed pooled Soft-clDice": 100*PROPOSED_REFERENCE["overall"]["soft_cldice_pooled"],
    "Proposed pooled Endpoint L2": PROPOSED_REFERENCE["overall"]["endpoint_l2_pooled"],
    "Proposed geometry coverage": 100*PROPOSED_REFERENCE["overall"]["endpoint_coverage"],
}
with open(OUT/"metrics"/"proposed_pooled_geometry_diagnostics.json","w",encoding="utf-8") as f:
    json.dump(pooled_geometry_diagnostics, f, indent=2)

baseline_family = {r["group"]:r for r in by_family}
family_compare = []
for fam in DATASET_FAMILIES:
    if fam not in baseline_family:
        continue
    b = baseline_family[fam]
    p = PROPOSED_REFERENCE[fam]
    family_compare.extend([
        {
            "family":fam,
            "method":"Arrow R-CNN common-protocol reimplementation",
            "Link F1":100*b["link_f1"],
            "nSEE":b["nSEE"],
            "Endpoint L2":b.get("endpoint_l2_macro", b["endpoint_l2_mean"]),
        },
        {
            "family":fam,
            "method":"Proposed framework",
            "Link F1":100*p["link_f1"],
            "nSEE":p["nSEE"],
            "Endpoint L2":p["endpoint_l2_macro"],
        },
    ])
write_rows(OUT/"metrics"/"comparison_by_family.csv", family_compare)


## 9. Visual sanity set

GT is drawn with thin dark marks; baseline predictions are drawn with thicker
light marks. These overlays are for manual sanity checking, not evaluation.

In [ ]:
if SAVE_DEBUG_OVERLAYS:
    debug_counts=Counter()
    for raw,record in zip(TEST_RAWS,TEST_RECORDS):
        sample=record["sample"]
        if debug_counts[sample.family]>=MAX_DEBUG_OVERLAYS_PER_FAMILY: continue
        pred=graph_from_detection(raw,sample,SELECTED_SCORE_THRESHOLD); gt=load_gt_graph(sample); img=cv2.imread(str(sample.image_path),cv2.IMREAD_COLOR)
        if img is None: continue
        for n_ in gt.get("nodes",[]):
            x,y,w,h=[int(round(v)) for v in n_["bbox"]]; cv2.rectangle(img,(x,y),(x+w,y+h),(0,0,0),1)
        for e in gt.get("arrows",[]):
            t,h_=point_from_edge(e,"tail"),point_from_edge(e,"head")
            if t and h_: cv2.arrowedLine(img,(int(round(t[0])),int(round(t[1]))),(int(round(h_[0])),int(round(h_[1]))),(0,0,0),1,tipLength=.08)
        for n_ in pred.get("nodes",[]):
            x,y,w,h=[int(round(v)) for v in n_["bbox"]]; cv2.rectangle(img,(x,y),(x+w,y+h),(255,255,255),2)
        for e in pred.get("arrows",[]):
            t,h_=point_from_edge(e,"tail"),point_from_edge(e,"head")
            if t and h_: cv2.arrowedLine(img,(int(round(t[0])),int(round(t[1]))),(int(round(h_[0])),int(round(h_[1]))),(255,255,255),2,tipLength=.08)
        out=OUT/"debug"/f"{sample.family}_{sample.sample_id}_overlay.png"; cv2.imwrite(str(out),img); debug_counts[sample.family]+=1
    print("Debug overlays:",dict(debug_counts))

## 10. Output checklist and manuscript fidelity statement

Main outputs:

- `checkpoints/arrow_rcnn_common_final.pt`
- `training_history.csv`
- `validation_threshold_sweep.csv`
- `selected_inference_config.json`
- `metrics/metrics_summary.json`
- `metrics/metrics_per_image.csv`
- `metrics/metrics_by_family.csv`
- `metrics/comparison_with_proposed.csv`
- `metrics/comparison_by_family.csv`
- `arrow_rcnn_graph_predictions.zip`

**Recommended paper wording:**

> We reimplemented Arrow R-CNN as an external baseline using its ResNet-101-FPN Faster R-CNN backbone and parallel RoI head/tail regression mechanism. For a controlled topology-recovery comparison, semantic node subclasses were collapsed into a single node class while text was retained as a distractor class during training. The baseline used the same reconstructed input diagrams, family-stratified learning split, and held-out test diagrams as the proposed framework. Its predicted node boxes and arrow endpoints were converted to directed graphs using nearest-node attachment and duplicate-edge suppression. Both systems were then evaluated with the same class-agnostic node matcher, directed-link protocol, and normalized structural edit error.

This should be described as an **Arrow R-CNN reimplementation**, not the
authors' official code. The original work used `maskrcnn-benchmark`; a public
third-party Detectron2 implementation was used as an architectural cross-check,
while this notebook uses current torchvision for runtime compatibility.

If `strong_30k` is used, report the actual 30k-step training schedule. Do
not describe it as the paper's 90k schedule. If the baseline is still improving
near 30k, resume training or run `paper_90k` before freezing manuscript values.

In [ ]:
print("\nArrow R-CNN common-protocol baseline finished.")
print("Output root:",OUT)
print("Selected score threshold:",SELECTED_SCORE_THRESHOLD)
print("Primary table:",OUT/"metrics"/"comparison_with_proposed.csv")
print("Family table :",OUT/"metrics"/"comparison_by_family.csv")
print("Checkpoint   :",OUT/"checkpoints"/"arrow_rcnn_common_final.pt")
print("Run mode     :",MODE)
if MODE=="evaluate_existing":
    print("Training     : SKIPPED")
    print("Loaded model :",RESOLVED_BASELINE_CHECKPOINT)
print("Endpoint convention:",ARROW_ENDPOINT_CONVENTION)
print("Direction audit:",OUT/"metrics"/"direction_semantics_check.json")
if REPORT_FIXED_PAPER_THRESHOLD:
    print("Fixed 0.70 metrics:",OUT/"metrics"/"metrics_fixed_threshold_0p70.json")

print("GPU count available:",GPU_COUNT)
print("Multi-GPU training used:",USE_MULTI_GPU)
if MODE=="train":
    print("Global physical batch:",physical_bs)
    print(
        "Per-GPU batch chunks:",
        _balanced_chunk_sizes(physical_bs,GPU_COUNT) if USE_MULTI_GPU else [physical_bs]
    )
print("DataLoader workers:",min(NUM_DATALOADER_WORKERS,os.cpu_count() or 1))

print_host_memory("final")
